# Cómputo y concurrencia · notebook de la unidad 11

Este notebook es el lugar **verbose** de la unidad: aquí cada paso se explica,
el código se repite entre versiones (cada versión trae completa la anterior)
y lo que cambia se marca con `# NUEVO:`. Las páginas del curso traen sólo las
celdas cortas `Haz (celda k.m)`; esas celdas aparecen aquí **literales**
(su primer renglón es `# celda k.m`), rodeadas de versiones expandidas.

Cada tema cierra con un `assert` (si falla, algo en tu ambiente o tu servidor
no es lo que la página supone) y una tabla impresa de resultados.

**Reglas de las celdas.** Usan `await` de nivel superior: el kernel de Jupyter
ya tiene un event loop corriendo, así que `asyncio.run(...)` truena aquí y
nunca aparece. Las cargas son pequeñas: a lo más 500 llamadas y a lo más
4 procesos de cálculo.

## 0.1 · Arranque: ¿está vivo el servidor y ves tus módulos?

La unidad mide contra el servidor de la clase (`servidor.py`, puerto 8026).
Debe estar corriendo en **otra terminal**, desde esta misma carpeta:

```text
FDD_SILENCIO=1 uv run python servidor.py
```

`FDD_SILENCIO=1` apaga la radiografía por petición: imprimir 500 peticiones
en una terminal es trabajo del servidor que estorbaría a la medición.

La celda de abajo comprueba cuatro cosas: que `cable.py`, `gantt.py` y
`trabajo_computo.py` se importan (viven junto a este notebook), que el
servidor contesta `200`, cuántas CPU lógicas tiene tu máquina (el tope `P` de
ejecución simultánea de la página 2) y las versiones de las bibliotecas.

In [ ]:
# 0.1 arranque
import os, sys, platform
import requests, httpx, numpy, matplotlib
from cable import URL
import gantt                       # el Gantt medido de este notebook
import trabajo_computo             # el cálculo de los procesos

r = requests.get(f"{URL}/", timeout=5)
print("servidor:", URL, "->", r.status_code)
print("CPU lógicas (P):", os.cpu_count())
print("Python", platform.python_version(), "|", "requests", requests.__version__,
      "| httpx", httpx.__version__, "| numpy", numpy.__version__,
      "| matplotlib", matplotlib.__version__)
assert r.status_code == 200, "arranca servidor.py en otra terminal"
assert os.cpu_count() >= 2, "con 1 CPU lógica no hay paralelismo que medir"

**Qué esperar.** `servidor: http://127.0.0.1:8026 -> 200`, el número de CPU
lógicas de tu máquina y las versiones. Si `requests.get` truena con
`ConnectionError`, el servidor no está corriendo.

## 0.2 · El Gantt medido: cómo se lee

`gantt.py` (junto a este notebook) guarda intervalos `[inicio, fin)` por
tarea y los dibuja. Es la herramienta de todo el notebook, así que se prueba
una vez con datos inventados a mano:

| tipo | se dibuja | significa |
|---|---|---|
| `exec` | relleno | la tarea ejecuta instrucciones en una CPU |
| `espera` | contorno | espera algo de fuera del programa (red, reloj) |
| `lista` | rayado | puede ejecutar pero aún no tiene hilo o CPU |
| `permiso` | ámbar | espera un permiso de otra tarea (semáforo, candado, pool) |

Los intervalos son semiabiertos: la barra siguiente empieza donde acaba la
anterior y no se solapan.

In [ ]:
# 0.2 prueba del Gantt con tiempos inventados (no mide nada)
gantt.reinicia()
for i, (ini, lista, esp) in enumerate([(0.0, 0.0, 0.2), (0.0, 0.1, 0.2)]):
    gantt.registra(f"tarea {i}", ini, ini + lista, "lista")
    gantt.registra(f"tarea {i}", ini + lista, ini + lista + 0.02, "exec")
    gantt.registra(f"tarea {i}", ini + lista + 0.02, ini + lista + 0.02 + esp, "espera")
gantt.registra("tarea 1", 0.34, 0.40, "permiso")
gantt.dibuja("Prueba: las cuatro formas")
assert len(gantt.registros()) == 7

**Qué esperar.** Dos filas: la segunda empieza con una barra rayada (lista),
luego un trazo relleno corto (exec), un contorno largo (espera) y al final un
tramo ámbar. Si ves las cuatro formas, `gantt.py` y matplotlib funcionan.
Cada tema empieza con `gantt.reinicia()` para no mezclar dibujos.

# Tema 1 · Dónde pasa el tiempo

**Pregunta.** Una llamada a una API tarda 0.2 s: ¿cuánto de eso trabaja tu
CPU? **Meta.** Medir, con dos relojes, que el tiempo de un cliente de red es
casi todo espera, y construir la **v1** del cliente (un `for` con `requests`)
midiéndola con un Gantt real.

Vocabulario de la página 1: una **tarea** pasa por estados; en `exec` usa CPU,
en `espera` aguarda algo de fuera (aquí, al servidor) y en `lista` podría
ejecutar pero no tiene CPU. `ρ = exec / vida` es la fracción de la vida que
la tarea de verdad trabaja.

## Paso 1 · Dos relojes

Python tiene dos relojes para un mismo programa:

- `time.perf_counter()`: el reloj de pared, segundos reales transcurridos.
- `time.process_time()`: segundos de CPU que **tu proceso** usó (excluye el
  tiempo que está dormido o esperando red).

Si el segundo es mucho menor que el primero, la tarea pasó casi todo su
tiempo en `espera`. La celda mide una llamada al servidor con
`?lento=0.2` (el servidor duerme 0.2 s antes de contestar) y, de contraste,
un bucle puro de Python que nunca espera nada.

**Qué esperar.** La llamada: reloj ≈ 0.2 s, CPU de unos milisegundos, CPU/reloj
de un solo dígito de por ciento. El bucle: reloj y CPU casi iguales, ≈ 100 %.

In [ ]:
# celda 1.1
import time, requests
from cable import URL

def cuenta(n=3_000_000):
    suma = 0
    for i in range(n): suma += i * i      # bucle puro: nunca espera
def mide(nombre, tarea):
    r0, c0 = time.perf_counter(), time.process_time()   # pared y CPU
    tarea()
    r, c = time.perf_counter() - r0, time.process_time() - c0
    print(f"{nombre:22} reloj {r:5.2f} s   CPU {c:5.2f} s   CPU/reloj {c / r:5.0%}")
mide("llamada ?lento=0.2", lambda: requests.get(f"{URL}/ventas?lento=0.2", timeout=5))
mide("bucle puro de Python", cuenta)

Lee el resultado como estados: la llamada fue `espera` casi toda su vida
(ρ ≈ 0); el bucle fue `exec` toda la suya (ρ ≈ 1). Esa diferencia decide
todo lo demás de la unidad: lo que **espera** se puede solapar con otra cosa;
lo que **ejecuta** sólo se acelera con más CPU.

## Paso 2 · Presupuesto de latencia de un endpoint

Antes de medir más, la misma idea con cifras de un endpoint típico (cifras
**supuestas**, no medidas, para practicar la cuenta): consulta a la BD 50 ms,
llamada a otra API 1.5 s, procesar el resultado 1 ms de CPU. La latencia es la
suma, y la pregunta útil es qué parte es CPU.

In [ ]:
presupuesto = [("consulta a la BD", 0.050, "espera"),
               ("llamada a otra API", 1.500, "espera"),
               ("procesar el resultado", 0.001, "exec")]
total = sum(d for _, d, _ in presupuesto)
print(f"{'paso':24} {'segundos':>9} {'% de la latencia':>17}  estado")
for nombre, d, estado in presupuesto:
    print(f"{nombre:24} {d:9.3f} {d / total:17.1%}  {estado}")
print(f"{'total':24} {total:9.3f}")
assert abs(sum(d for _, d, e in presupuesto if e == "espera") / total - 0.9994) < 1e-3

**Qué esperar.** El 99.9 % de la latencia es `espera`. Hacer el CPU 10 veces
más rápido ahorra menos de un milisegundo; solapar la espera de la API con
otra cosa es lo que mueve la aguja. «Mide antes de elegir.»

## Paso 3 · Veinte llamadas en secuencia

La celda 1.2 de la página repite la llamada 20 veces con un `for` y reparte el
tiempo entre CPU y espera con los mismos dos relojes. Un `for` que espera la
respuesta de una llamada antes de hacer la siguiente es **secuencial**: la
vida de una tarea empieza justo donde acaba la anterior. Predicción antes de
correr: 20 × 0.2 s = 4 s de sueño pedido, más algo de sobrecarga de red.

In [ ]:
# celda 1.2
import time, requests
from cable import URL

N = 20
r0, c0 = time.perf_counter(), time.process_time()
for _ in range(N):                          # una llamada tras otra: secuencial
    requests.get(f"{URL}/ventas?lento=0.2", timeout=5)
r, c = time.perf_counter() - r0, time.process_time() - c0
print(f"{N} llamadas en serie: {r:.2f} s  ({r / N:.3f} s por llamada)")
print(f"ejecuta (CPU):  {c:.2f} s  ({c / r:.1%})")
print(f"espera:         {r - c:.2f} s  ({(r - c) / r:.1%})")
print(f"sueño pedido al servidor: {N * 0.2:.1f} s")

**Qué pasó.** `espera` ≈ 99 % del reloj; `ejecuta (CPU)` ≈ 1 %. El reloj es
un poco mayor que 4.0 s: cada llamada paga además unos milisegundos de red,
armar la petición y leer la respuesta.

## Paso 4 · v1 del cliente, medida con un Gantt real

Esta es la **v1** del cliente que la unidad va a mejorar en cada tema. Es el
mismo `for` de la celda anterior, con una línea marcada `# NUEVO:` que anota
en `gantt` cuándo empezó y cuándo terminó cada llamada. Anotamos cada llamada
completa como `espera`: desde que sale la petición hasta que llega la
respuesta, el cliente no trabaja (CPU ≈ 1 %, lo medido arriba). Cada llamada
es una tarea (una fila).

In [ ]:
# v1: un for con requests, una llamada tras otra
import time, requests
from cable import URL
import gantt                                          # NUEVO: el Gantt medido

def cliente_v1(n, lento=0.2):
    gantt.reinicia()                                  # NUEVO: dibujo limpio
    codigos = []
    inicio = time.perf_counter()
    for i in range(n):                                # una llamada tras otra
        a = time.perf_counter()                       # NUEVO: cuándo sale la petición
        r = requests.get(f"{URL}/ventas?lento={lento}", timeout=5)
        b = time.perf_counter()                       # NUEVO: cuándo llega la respuesta
        gantt.registra(f"llamada {i:02d}", a, b, "espera")   # NUEVO: anota el intervalo
        codigos.append(r.status_code)
    return codigos, time.perf_counter() - inicio

codigos, t_v1 = cliente_v1(20)
gantt.dibuja("v1 · 20 llamadas en serie (for + requests)")

**Qué esperar.** Una escalera: 20 filas, cada barra de contorno (espera)
empieza donde termina la anterior y a ningún instante hay dos barras
abiertas a la vez. La anchura total del dibujo es `T`, la suma de las 20
duraciones. Eso es lo que significa **secuencial**.

## Paso 5 · Resultados del tema 1

`assert`s: todas las respuestas fueron `200`, no hubo solapamiento (cada
llamada empieza donde o después de donde acaba la anterior) y el tiempo total
cubre el sueño pedido. Después, una pequeña tabla que irá creciendo versión
por versión en los temas siguientes.

In [ ]:
regs = sorted(gantt.registros(), key=lambda d: d[1])
assert set(codigos) == {200}
assert all(regs[k + 1][1] >= regs[k][2] for k in range(len(regs) - 1)), "hay solape: no es secuencial"
assert t_v1 >= 20 * 0.2

print(f"{'versión':30} {'llamadas':>8} {'tiempo (s)':>11} {'llam/s':>8}")
print(f"{'v1 for + requests':30} {20:8d} {t_v1:11.2f} {20 / t_v1:8.1f}")
print(f"predicción para 500 llamadas en serie: {500 * t_v1 / 20:6.0f} s (no se corre: {500 * 0.2:.0f} s de pura espera)")

# Tema 2 · Secuencial, concurrente, paralelo

**Pregunta.** ¿Cómo bajar las 4 s de la v1? Dos respuestas distintas según
dónde se va el tiempo: si **espera**, se solapan las esperas (hilos); si
**ejecuta**, hacen falta más CPU (procesos). **Meta.** La v2 del cliente con
`ThreadPoolExecutor`, el cálculo con 1, 2 y 4 procesos con su gráfica de
speedup, y la carrera de datos paso a paso.

Definiciones de la página 2, sobre la traza (el Gantt):

- **secuencial:** nunca hay dos tareas con vida simultánea.
- **concurrente:** hay dos tareas cuyas vidas se solapan.
- **paralelo:** hay dos tareas **ejecutando** a la vez (necesita `P ≥ 2` CPU).
  Todo lo paralelo es concurrente, no al revés.

## Paso 1 · Las 20 llamadas con 32 hilos

`ThreadPoolExecutor(max_workers=32)` crea hasta 32 hilos del mismo proceso.
`hilos.map(pide, range(20))` entrega las 20 llamadas al pool: cada hilo que
queda libre toma la siguiente. Mientras un hilo está dentro de
`requests.get` esperando red, **suelta el GIL** (el candado del intérprete) y
otro hilo puede correr. Con 20 llamadas y 32 hilos, todas empiezan casi a la
vez y la espera es la de **una** llamada.

**Qué esperar.** Un resultado de ≈ 0.25 s en lugar de ≈ 4 s, con `status {200}`.
Es concurrente (las vidas se solapan) pero no es paralelo: nadie ejecuta a la
vez con nadie, sólo esperan a la vez.

In [ ]:
# celda 2.1
import time
import requests
from concurrent.futures import ThreadPoolExecutor
from cable import URL                      # http://127.0.0.1:8026, el servidor de la clase

def pide(i):                               # el servidor duerme 0.2 s y contesta
    return requests.get(f"{URL}/ventas?lento=0.2", timeout=5).status_code

inicio = time.perf_counter()
with ThreadPoolExecutor(max_workers=32) as hilos:   # hasta 32 hilos del mismo proceso
    codigos = list(hilos.map(pide, range(20)))      # lanza las 20 sin esperar a la anterior
print(f"{len(codigos)} llamadas, status {set(codigos)}, {time.perf_counter() - inicio:.2f} s")

`codigos` es la lista de los 20 status; `set(codigos)` los colapsa a los
valores distintos y debe ser `{200}`.

In [ ]:
assert set(codigos) == {200}
t_celda_21 = time.perf_counter() - inicio
print(f"tiempo de la celda 2.1: {t_celda_21:.2f} s")

## Paso 2 · v2 del cliente, medida: dónde espera cada llamada

La v2 toma el cliente v1 completo y cambia sólo el bucle: ahora cada llamada
es una función que se **envía** al pool con `submit`. Lo nuevo para el Gantt:
además de `espera` (dentro de `requests.get`), registramos `lista` desde que
la llamada se envía hasta que un hilo la empieza a correr. Con 32 hilos esa
franja es casi cero; con **4** hilos, 20 llamadas hacen cinco olas y las
llamadas de la segunda ola en adelante pasan antes un rato en `lista`
(rayado). Predicción para `k` hilos: `⌈20/k⌉ × 0.2 s`.

In [ ]:
# v2: el cliente v1, con un ThreadPoolExecutor
import time, requests
from concurrent.futures import ThreadPoolExecutor      # NUEVO: el pool de hilos
from cable import URL
import gantt

def pide_v2(i, enviada, lento=0.2):                    # NUEVO: la llamada es una función
    empieza = time.perf_counter()                      # NUEVO: cuándo un hilo la toma
    gantt.registra(f"llamada {i:02d}", enviada, empieza, "lista")   # NUEVO: esperó turno
    r = requests.get(f"{URL}/ventas?lento={lento}", timeout=5)
    gantt.registra(f"llamada {i:02d}", empieza, time.perf_counter(), "espera")
    return r.status_code

def cliente_v2(n, hilos, lento=0.2):
    gantt.reinicia()
    inicio = time.perf_counter()
    with ThreadPoolExecutor(max_workers=hilos) as pool:      # NUEVO: hasta `hilos` a la vez
        futuros = [pool.submit(pide_v2, i, time.perf_counter(), lento)   # NUEVO: submit, no map
                   for i in range(n)]
        codigos = [f.result() for f in futuros]              # f.result() espera a ese futuro
    return codigos, time.perf_counter() - inicio

codigos4, t_h4 = cliente_v2(20, 4)
gantt.dibuja("v2 · 20 llamadas, 4 hilos")
codigos32, t_h32 = cliente_v2(20, 32)
gantt.dibuja("v2 · 20 llamadas, 32 hilos")

**Qué esperar.** Con 4 hilos: cinco olas de 4 barras de contorno, y las
filas de las olas 2 a 5 empiezan con un trazo rayado (esperaron turno). Con 32:
20 barras casi alineadas arriba y abajo, sin rayado: todas arrancaron a la
vez. Cada barra dura lo mismo (0.2 s): el hilo no hizo la llamada más rápida,
la **solapó** con las otras.

In [ ]:
assert set(codigos4) == set(codigos32) == {200}
assert t_h32 < t_h4 < 20 * 0.2          # más hilos, menos tiempo; ambos mejor que la v1
print(f"{'versión':30} {'llamadas':>8} {'tiempo (s)':>11} {'predicción (s)':>15} {'vs v1':>7}")
print(f"{'v1 for + requests':30} {20:8d} {t_v1:11.2f} {20 * 0.2:15.2f} {'1.0x':>7}")
for nombre, t, k in (("v2 hilos (k=4)", t_h4, 4), ("v2 hilos (k=32)", t_h32, 32)):
    pred = -(-20 // k) * 0.2
    print(f"{nombre:30} {20:8d} {t:11.2f} {pred:15.2f} {t_v1 / t:6.1f}x")

## Paso 3 · Cálculo: 4 tareas con 1, 2 y 4 procesos

Un cálculo puro de CPU es lo opuesto: no hay espera que solapar. Con hilos
no ayudaría (el GIL deja ejecutar **un** hilo de Python a la vez por
intérprete). Los **procesos** sí: cada uno tiene su propio intérprete y su
propio GIL, y el sistema operativo los reparte entre las CPU.

`ProcessPoolExecutor` manda a cada hijo el *nombre* de la función (módulo +
nombre), no su código; por eso `calcula` vive en `trabajo_computo.py` y no en
la celda. `mide(n)` incluye arrancar los procesos, que es parte del costo real.

**Qué esperar.** 4 tareas idénticas: con 1 proceso T₁, con 2 aproximadamente
T₁/2, con 4 aproximadamente T₁/4 (menos si tu máquina tiene menos de 4 CPU o
está ocupada). Cuanto más cerca de la mitad y el cuarto, menos serial hay.

In [ ]:
# celda 2.2
import os
import time
from concurrent.futures import ProcessPoolExecutor
from trabajo_computo import calcula        # vive en un archivo: el proceso hijo la importa

def mide(n_procesos):
    inicio = time.perf_counter()
    with ProcessPoolExecutor(max_workers=n_procesos) as procesos:
        list(procesos.map(calcula, [5_000_000] * 4))   # 4 trabajos idénticos de sólo CPU
    return time.perf_counter() - inicio                # incluye arrancar los procesos

print("CPU lógicas:", os.cpu_count())      # el tope P de ejecución simultánea
for n in (1, 2, 4):
    print(f"{n} proceso(s): {mide(n):.2f} s")

## Paso 4 · Los mismos procesos, con Gantt y gráfica de speedup

Repetimos el cálculo con `calcula_medido` (una función nueva en
`trabajo_computo.py`, que devuelve cuándo empezó y terminó con `time.time()`:
los procesos necesitan un reloj común). Cada tarea se dibuja como `exec`; con
menos procesos que tareas, las que esperan una CPU libre salen como `lista`
(desde el `submit` hasta que un proceso la toma).

El **speedup** es `S(N) = T₁ / T_N` y la **eficiencia** `E(N) = S(N) / N`.
Con `E` cerca de 1 los procesos se reparten el trabajo sin estorbarse.

In [ ]:
# v2b: el cálculo en procesos, medido por tarea
import time
from concurrent.futures import ProcessPoolExecutor
from trabajo_computo import calcula_medido             # NUEVO: la versión que devuelve sus tiempos
import gantt

def mide_procesos(n_procesos, tareas=4, n=5_000_000):
    gantt.reinicia()
    inicio = time.perf_counter()
    with ProcessPoolExecutor(max_workers=n_procesos) as procesos:
        enviada = time.time()                                       # NUEVO: reloj común (pared)
        futuros = [procesos.submit(calcula_medido, n) for _ in range(tareas)]
        tiempos = [f.result() for f in futuros]                     # (inicio, fin, resultado)
    total = time.perf_counter() - inicio
    for i, (a, b, _) in enumerate(tiempos):                         # NUEVO: una fila por tarea
        gantt.registra(f"tarea {i}", enviada, a, "lista")
        gantt.registra(f"tarea {i}", a, b, "exec")
    return total, tiempos

tiempos_n = {}
for k in (1, 2, 4):
    tiempos_n[k], _ = mide_procesos(k)
    gantt.dibuja(f"4 tareas de cálculo con {k} proceso(s)")

**Qué esperar.** Con 1 proceso: una escalera de 4 barras rellenas, con las
tres últimas precedidas de un trazo rayado (lista: esperando que el único
proceso quede libre). Con 4 procesos: cuatro barras rellenas casi alineadas.
Esas cuatro barras alineadas son lo que hace **paralelo** a un Gantt: dos
`exec` en el mismo instante.

In [ ]:
import matplotlib.pyplot as plt
Ns = (1, 2, 4)
S = {k: tiempos_n[1] / tiempos_n[k] for k in Ns}
fig, ax = plt.subplots(figsize=(5, 3.4))
ax.plot(Ns, [S[k] for k in Ns], "o-", color="#1f4e8c", label="medido")
ax.plot(Ns, Ns, "--", color="#555555", label="ideal S(N) = N")
ax.set_xticks(Ns); ax.set_xlabel("procesos N"); ax.set_ylabel("speedup S(N)")
ax.set_title("Speedup del cálculo en procesos"); ax.legend(); fig.tight_layout(); plt.show()

print(f"{'procesos N':>10} {'T_N (s)':>9} {'S(N)':>6} {'E(N)':>6}")
for k in Ns:
    print(f"{k:10d} {tiempos_n[k]:9.2f} {S[k]:6.2f} {S[k] / k:6.2f}")
assert tiempos_n[4] < tiempos_n[1], "4 procesos deberían ganarle a 1 si tienes ≥ 2 CPU libres"
assert all(0 < S[k] <= k * 1.15 for k in Ns), "el speedup no puede pasar de N (con margen de ruido)"

**Cómo leer la tabla.** `S(N)` casi siempre queda un poco debajo de `N`: arrancar
procesos, mandar argumentos y recoger resultados es trabajo que no se
reparte (la fracción serial `f` de la ley de Amdahl, página 6). Con
`E(N)` bajando a medida que sube `N`, agregar procesos rinde cada vez menos.

## Paso 5 · La carrera de datos, paso a paso (determinista)

`contador += 1` no es una operación indivisible: son tres pasos
(**leer**, **sumar**, **escribir**) y el intérprete puede cambiar de hilo
entre dos cualesquiera. Si A y B leen el mismo valor antes de que alguno
escriba, las dos suman sobre el valor viejo y una actualización se pierde.

Para verlo sin esperar a que ocurra por azar, esta celda simula dos hilos con
generadores (cada `yield` es un posible cambio de hilo) y fija a mano el
orden: A lee, B lee, A escribe, B escribe.

In [ ]:
def hilo_sim(nombre, memoria):                       # un "hilo" con 3 pasos, cada yield es un cambio posible
    actual = memoria["contador"];  yield f"{nombre} lee {actual}"
    nuevo = actual + 1;            yield f"{nombre} calcula {nuevo}"
    memoria["contador"] = nuevo;   yield f"{nombre} escribe {nuevo}"

memoria = {"contador": 0}
a, b = hilo_sim("A", memoria), hilo_sim("B", memoria)
orden = [a, b, a, b, a, b]                           # A1 B1 A2 B2 A3 B3: el peor entrelazado
print(f"{'paso':>4}  {'qué pasa':20} {'contador en memoria':>20}")
for k, g in enumerate(orden, 1):
    texto = next(g)
    print(f"{k:4d}  {texto:20} {memoria['contador']:20d}")
print("dos incrementos, resultado:", memoria["contador"])
assert memoria["contador"] == 1, "con este entrelazado se pierde una actualización"

**Qué pasó.** A y B leyeron `0`; A escribió `1`; B también escribió `1`. Se
hicieron dos incrementos y el contador subió una vez. Con `sys.setswitchinterval`
muy pequeño, la celda siguiente fuerza ese mismo tipo de entrelazado con hilos
reales: 4 hilos × 100 000 incrementos deberían dar 400 000.

In [ ]:
# celda 2.3
import sys, threading
sys.setswitchinterval(1e-6)          # turno cada 1 µs (de fábrica, 5 ms): la carrera aparece casi siempre
contador = 0
def suma_uno(x):                     # la llamada abre un punto donde el intérprete puede cambiar de hilo
    return x + 1
def suma(veces):
    global contador                  # los 4 hilos escriben la misma variable
    for _ in range(veces):
        actual = contador            # 1. leer
        contador = suma_uno(actual)  # 2-3. sumar y escribir
hilos = [threading.Thread(target=suma, args=(100_000,)) for _ in range(4)]
for h in hilos: h.start()
for h in hilos: h.join()             # espera a que terminen los cuatro
print(f"esperado {4 * 100_000:,}, obtenido {contador:,}")
sys.setswitchinterval(0.005)         # regresa al valor de fábrica para las celdas siguientes

El resultado **cambia entre corridas** (es no determinista): depende de
cuándo el intérprete cambia de hilo. Repítelo y verás otro `obtenido`, casi
siempre menor que `esperado`. Ahora se mide cuántas actualizaciones se
pierden en varias repeticiones, y se repite con el candado de la celda 2.4.
Un `Lock` hace la **región crítica** (leer + escribir) indivisible: los hilos
hacen fila para entrar, y se pierde paralelismo a propósito, pero no datos.

In [ ]:
perdidas_sin = 400_000 - contador
print("perdidas en la corrida de la celda 2.3:", f"{perdidas_sin:,}")

In [ ]:
# celda 2.4
import sys, threading
sys.setswitchinterval(1e-6)          # el mismo forzado de la celda 2.3
contador, candado = 0, threading.Lock()
def suma_uno(x):
    return x + 1
def suma_con_candado(veces):
    global contador
    for _ in range(veces):
        with candado:                # región crítica: leer y escribir sin que otro hilo se cuele
            contador = suma_uno(contador)
hilos = [threading.Thread(target=suma_con_candado, args=(100_000,)) for _ in range(4)]
for h in hilos: h.start()
for h in hilos: h.join()
print(f"esperado {4 * 100_000:,}, obtenido {contador:,}")
sys.setswitchinterval(0.005)         # regresa al valor de fábrica

**Qué esperar.** `esperado 400,000, obtenido 400,000`. Siempre. El candado no
cuesta casi nada aquí porque la región crítica es corta; si la región crítica
fuera todo el trabajo, los hilos quedarían secuenciales.

In [ ]:
assert contador == 4 * 100_000
print(f"{'caso':26} {'esperado':>10} {'obtenido':>10} {'perdidas':>9}")
print(f"{'sin candado (celda 2.3)':26} {400_000:10,} {400_000 - perdidas_sin:10,} {perdidas_sin:9,}")
print(f"{'con Lock (celda 2.4)':26} {400_000:10,} {contador:10,} {400_000 - contador:9,}")

# Tema 3 · Asíncrono y el event loop

**Pregunta.** Los hilos solapan esperas, pero cada hilo cuesta memoria y el
sistema los reparte. ¿Se pueden solapar esperas en **un** solo hilo?
**Meta.** Entender el mecanismo (corrutina, `Task`, event loop tick a tick)
y medir con Gantt reales qué pasa cuando se usa bien (`gather`), cuando se
usa mal (`await` en serie) y cuando se rompe (`time.sleep` dentro de una
corrutina).

**Asíncrono** (definición de la página 3) es un mecanismo: la llamada
**regresa antes de tener el resultado** y quien espera **cede el hilo**. No es
una celda del Gantt: es la manera en que una tarea deja de ocupar el hilo
mientras espera.

## Paso 1 · Corrutina y `Task`

- Una función `async def` no corre al llamarla: devuelve un objeto
  **corrutina** (una receta pausable).
- `asyncio.create_task(c)` agenda esa corrutina en el event loop y regresa de
  inmediato un `Task`; la corrutina empieza a correr la próxima vez que el
  loop recupere el control.
- `await t` suspende a quien lo escribe hasta que `t` termina y **cede el
  hilo** al loop mientras tanto.

**Qué esperar.** Primero `coroutine` (nada corrió); luego `terminó: False`
justo después de `create_task`; y al final `terminó: True y devolvió 42`.

In [ ]:
# celda 3.1
import asyncio

async def cuenta():                        # corrutina: puede suspenderse en un await
    await asyncio.sleep(0.2)               # espera 0.2 s cediendo el hilo al loop
    return 42

c = cuenta()                               # sólo crea el objeto: no corre nada
print("llamarla devuelve:", type(c).__name__)
c.close()                                  # se descarta sin correrla (evita el aviso)
tarea = asyncio.create_task(cuenta())      # la agenda en la cola de listas y regresa
print("justo después de crearla, terminó:", tarea.done())
resultado = await tarea                    # main cede el hilo hasta que la tarea termina
print("después de await, terminó:", tarea.done(), "y devolvió", resultado)

Aviso para el notebook: el kernel ya tiene un event loop; por eso la celda usa
`await` directo y nunca `asyncio.run`.

In [ ]:
assert tarea.done() and resultado == 42

## Paso 2 · El event loop tick a tick (simulación determinista)

Un **tick** del loop hace tres cosas, en este orden:

1. Corre, una tras otra, todas las tareas de la **cola de listas**, cada una
   hasta su siguiente `await` (o hasta que termina).
2. Pregunta al sistema qué esperas ya se cumplieron (`select`), con un
   plazo máximo igual al temporizador más próximo.
3. Pasa a la cola de listas las tareas cuya espera se cumplió.

La celda siguiente es un loop de juguete en Python puro (reloj **virtual**,
no mide nada real) con `gather(A, B, C)`: cada tarea ejecuta 10 ms, duerme
`d` y ejecuta 10 ms más, con `d_A = 0.3`, `d_B = 0.1`, `d_C = 0.2` s.
Imprime una fila por tick con las tres colas.

In [ ]:
import heapq
from collections import deque

def tarea_juguete(nombre, d):                  # cada yield es un await: cede el hilo al loop
    yield ("duerme", d)                        # entre el arranque y el yield, 10 ms de exec
    return nombre                              # después del yield, otros 10 ms de exec

def loop_juguete(spec, ms_exec=10):
    listas = deque((n, tarea_juguete(n, d)) for n, d in spec)    # la cola de listas
    durmiendo = []                                               # montículo: (vence_ms, nombre, tarea)
    reloj, tick, orden = 0, 0, []
    print("tick  reloj ms  cola de listas -> ejecutó  | esperando (vence en ms)")
    while listas or durmiendo:
        en_cola = ", ".join(n for n, _ in listas)                 # la cola de listas al empezar el tick
        corridas = []
        while listas:                                            # 1) corre toda la cola de listas
            nombre, g = listas.popleft()
            reloj += ms_exec                                     # su tramo de exec: el hilo ocupado
            try:
                _, d = next(g)                                   # corre hasta su await
                heapq.heappush(durmiendo, (reloj + round(d * 1000), nombre, g))
                corridas.append(nombre + " (hasta await)")
            except StopIteration:
                orden.append(nombre); corridas.append(nombre + " (termina)")
        esperando = ", ".join(f"{n}({v})" for v, n, _ in sorted(durmiendo))
        print(f"{tick:4d} {reloj:8d}  {en_cola:12} -> {', '.join(corridas)}  | esperando: {esperando}")
        if durmiendo:                                            # 2) select: salta al temporizador más próximo
            reloj = max(reloj, durmiendo[0][0])
            while durmiendo and durmiendo[0][0] <= reloj:        # 3) las vencidas pasan a listas
                _, nombre, g = heapq.heappop(durmiendo)
                listas.append((nombre, g))
        tick += 1
    return reloj, orden

fin_virtual, orden_fin = loop_juguete([("A", 0.3), ("B", 0.1), ("C", 0.2)])
print("terminaron en orden:", orden_fin, "| reloj virtual al final:", fin_virtual, "ms")
assert orden_fin == ["B", "C", "A"] and fin_virtual == 320

**Qué pasó.** El tick 0 corre A, B y C seguidas (cada una 10 ms, el hilo
ocupado: 30 ms) y las tres quedan esperando con su temporizador (A vence en
310, B en 120, C en 230). Entonces el loop no tiene nada que correr: pregunta
al sistema y **duerme hasta 120 ms**, el temporizador más próximo. Despierta B,
la corre y así sucesivamente. El tiempo total (320 ms) es el de la tarea más
larga (310) más lo que tarda en ejecutar el último tramo, no la suma de las tres.

Esa cola de listas y esa espera con plazo son todo el mecanismo. Lo que sigue
es el loop **real** de `asyncio`, instrumentado para que imprima su propio
número de tick.

## Paso 3 · La traza real de `gather(A, B, C)`

Dos celdas de la página. La primera define `marca`, que imprime cada evento
con el número de tick (un tick nuevo empieza cuando se agenda con
`call_soon`, que entra a la cola del tick siguiente) y los milisegundos desde
`t0`. La segunda corre `gather` con las tres tareas de la simulación, esta vez
con `asyncio.sleep` de verdad y `time.sleep(0.01)` como 10 ms de CPU ocupada.

In [ ]:
# celda 3.2
import asyncio, time
loop = asyncio.get_running_loop()          # el loop que corre las celdas

def abre_tick():                           # corre al principio del tick siguiente
    global en_tick
    en_tick = False

def marca(texto):                          # imprime: tick, ms desde t0, texto
    global tick, en_tick
    if not en_tick:                        # primera línea de este tick: le toca número nuevo
        tick, en_tick = tick + 1, True
        loop.call_soon(abre_tick)          # call_soon: entra a la cola del tick siguiente
    print(f"tick {tick}  {(loop.time() - t0) * 1000:4.0f} ms  {texto}")

In [ ]:
# celda 3.3
async def tarea(nombre, d):
    marca(f"{nombre} ejecuta 10 ms y hace await sleep({d})")
    time.sleep(0.01)                       # 10 ms con el hilo ocupado, sin ceder
    await asyncio.sleep(d)                 # cede el hilo; su temporizador vence en d s
    marca(f"{nombre} se reanuda, ejecuta 10 ms y termina")
    time.sleep(0.01)
    return nombre

t0, tick, en_tick = loop.time(), -1, False # reinicia el reloj y el contador
marca("main: gather crea A, B y C y hace await")
resultado = await asyncio.gather(tarea("A", 0.3), tarea("B", 0.1), tarea("C", 0.2))
marca(f"main recibe {resultado}")

**Qué esperar.** Los eventos de arranque de A, B y C salen en el mismo tick y
separados ~10 ms en el reloj (el hilo está ocupado con cada uno); luego B, C y
A se reanudan en ticks distintos, hacia los 120, 230 y 310 ms, y
`main recibe ['A', 'B', 'C']`: `gather` devuelve los resultados en el orden en
que se pasaron las corrutinas, no en el orden en que terminaron.

In [ ]:
assert resultado == ["A", "B", "C"]

## Paso 4 · `await` bien y `await` en serie, con Gantt medido

Misma tarea (llamadas de 0.2 s al servidor) con dos formas de escribirla:

- **`await` en serie:** `for ...: await cliente.get(...)`. Cada `await` cede el
  hilo, pero el `for` no lanza la siguiente llamada hasta tenerla resuelta. Es
  asíncrono (cede el hilo al esperar) y **secuencial** (nada se solapa).
- **`gather`:** crea todas las tareas primero y las espera juntas. Eso solapa.

Primero la celda de la página (serie), luego la versión **v3** del cliente.

In [ ]:
# celda 3.4
import time, httpx
from cable import URL

inicio = time.perf_counter()
async with httpx.AsyncClient(timeout=30) as cliente:    # cliente HTTP que cede el hilo al esperar
    for _ in range(20):
        r = await cliente.get(f"{URL}/ventas?lento=0.2")  # espera ésta antes de lanzar la siguiente
print(f"20 llamadas con await en serie: {time.perf_counter() - inicio:.2f} s")

**Qué esperar.** ≈ 4 s: igual que la v1 con `for` + `requests`. Que sea
`async` y use `await` no lo hizo concurrente.

In [ ]:
t_serie = time.perf_counter() - inicio
assert t_serie >= 20 * 0.2

### v3: corrutinas con `gather`, medida

Mismo cliente que antes, ahora como corrutina. Lo nuevo es `gather`, que
agenda las 20 llamadas como `Task` antes de esperar a ninguna, y el registro
del Gantt con `loop.time()`.

In [ ]:
# v3: el cliente v1/v2, con corrutinas y gather
import asyncio, time, httpx
from cable import URL
import gantt

async def pide_v3(cliente, i, lento=0.2):               # NUEVO: async def
    a = loop.time()
    r = await cliente.get(f"{URL}/ventas?lento={lento}")      # NUEVO: await cede el hilo
    gantt.registra(f"llamada {i:02d}", a, loop.time(), "espera")
    return r.status_code

async def cliente_v3(n, en_serie=False):
    gantt.reinicia()
    inicio = loop.time()
    async with httpx.AsyncClient(timeout=30) as cliente:      # NUEVO: cliente que cede el hilo
        if en_serie:                                          # el error: await dentro del for
            codigos = [await pide_v3(cliente, i) for i in range(n)]
        else:                                                 # NUEVO: gather agenda todas y espera juntas
            codigos = await asyncio.gather(*(pide_v3(cliente, i) for i in range(n)))
    return list(codigos), loop.time() - inicio

loop = asyncio.get_running_loop()
codigos_s, t_s = await cliente_v3(20, en_serie=True)
gantt.dibuja("v3 · 20 llamadas con await en serie")
codigos_g, t_g = await cliente_v3(20)
gantt.dibuja("v3 · 20 llamadas con gather")

**Qué esperar.** El primer dibujo es la escalera de la v1 (ninguna barra
empieza antes de que acabe la anterior). El segundo son 20 barras alineadas,
todas abiertas al mismo tiempo: concurrente, en **un** solo hilo. Esa es la
diferencia que hace `gather` y que no hace el `for` con `await`.

In [ ]:
assert set(codigos_s) == set(codigos_g) == {200}
assert t_s >= 20 * 0.2 and t_g < 1.0
print(f"{'versión':32} {'llamadas':>8} {'tiempo (s)':>11} {'vs v1':>7}")
print(f"{'v1 for + requests':32} {20:8d} {t_v1:11.2f} {'1.0x':>7}")
print(f"{'v2 hilos (k=32)':32} {20:8d} {t_h32:11.2f} {t_v1 / t_h32:6.1f}x")
print(f"{'v3 await en serie':32} {20:8d} {t_s:11.2f} {t_v1 / t_s:6.1f}x")
print(f"{'v3 gather':32} {20:8d} {t_g:11.2f} {t_v1 / t_g:6.1f}x")

## Paso 5 · `time.sleep` dentro de una corrutina rompe el loop

El loop solapa esperas **sólo si quien espera cede el hilo**. `await
asyncio.sleep(d)` lo cede; `time.sleep(d)` no: duerme el hilo entero y el loop,
que corre en ese hilo, no puede avanzar a ninguna otra tarea. La celda de la
página repite la traza de `gather(A, B, C)` pero con `time.sleep`.

**Qué esperar.** Las tareas corren **una tras otra** y el tiempo total es la
suma de las tres esperas más sus tramos: ≈ 0.3 + 0.1 + 0.2 + 0.06 = 0.66 s,
en lugar de ≈ 0.32 s. Ninguna se reanuda en otro tick: la `Task` B no puede
empezar hasta que A suelta el hilo al terminar.

In [ ]:
# celda 3.5
async def tarea_bloqueante(nombre, d):
    marca(f"{nombre} ocupa el hilo 10 ms + time.sleep({d}) + 10 ms")
    time.sleep(d + 0.02)                   # duerme SIN soltar el hilo: el loop no recupera el turno
    marca(f"{nombre} termina")
    return nombre

t0, tick, en_tick = loop.time(), -1, False
marca("main: gather crea A, B y C y hace await")
tareas = [("A", 0.3), ("B", 0.1), ("C", 0.2)]
resultado = await asyncio.gather(*(tarea_bloqueante(n, d) for n, d in tareas))
marca(f"main recibe {resultado}")

### La misma comparación, medida con Gantt

Las tres tareas de la traza con `await asyncio.sleep` y con `time.sleep`, cada
una con sus tres tramos: `exec` 10 ms, `espera` (o la espera que bloquea el
hilo), `exec` 10 ms. Lo nuevo: las tareas que todavía no pueden correr porque
otra tiene el hilo se registran como `lista` (rayado) desde que `gather` las
creó hasta que arrancan.

In [ ]:
async def tarea_g(nombre, d, bloqueante):
    inicio = loop.time()
    gantt.registra(nombre, T0, inicio, "lista")           # NUEVO: lista hasta que el hilo es suyo
    time.sleep(0.01)                                      # 10 ms de exec (hilo ocupado)
    a = loop.time(); gantt.registra(nombre, inicio, a, "exec")
    if bloqueante:
        time.sleep(d)                                     # NUEVO: duerme el hilo del loop
    else:
        await asyncio.sleep(d)                            # cede el hilo
    b = loop.time(); gantt.registra(nombre, a, b, "espera")
    time.sleep(0.01)                                      # 10 ms de exec
    gantt.registra(nombre, b, loop.time(), "exec")

async def corre_tres(bloqueante):
    global T0
    gantt.reinicia()
    T0 = loop.time()
    await asyncio.gather(*(tarea_g(n, d, bloqueante) for n, d in (("A", 0.3), ("B", 0.1), ("C", 0.2))))
    return loop.time() - T0

t_async = await corre_tres(False); gantt.dibuja("tres tareas con await asyncio.sleep")
t_bloq = await corre_tres(True);   gantt.dibuja("tres tareas con time.sleep (bloquea el loop)")

**Qué esperar.** Primer dibujo: las tres filas comparten el intervalo de espera
(contornos alineados), total ≈ 0.32 s. Segundo: las tres tareas ejecutan una
tras otra, y las que esperan turno aparecen como `lista` (rayadas) mientras
la otra duerme el hilo; el total es la suma, ≈ 0.66 s. El mismo código con
**una** palabra distinta (`time.sleep` por `await asyncio.sleep`) pasa de
concurrente a secuencial.

In [ ]:
assert t_async < 0.45 and t_bloq > 0.6
assert t_bloq > 1.5 * t_async
print(f"{'tres tareas A=0.3 B=0.1 C=0.2':32} {'tiempo (s)':>11} {'predicción (s)':>15}")
print(f"{'await asyncio.sleep':32} {t_async:11.2f} {0.3 + 0.02:15.2f}")
print(f"{'time.sleep (bloquea el loop)':32} {t_bloq:11.2f} {0.3 + 0.1 + 0.2 + 0.06:15.2f}")

# Tema 4 · El cliente de 500 llamadas, en cinco versiones

**Pregunta.** Tienes que hacer 500 llamadas de 0.2 s a la misma dependencia. ¿Cuánto tardan, y qué cambia de
una versión del cliente a la siguiente?

Este tema es el lugar verboso de la página 4 (página 4, concurrencia acotada): aquí el mismo cliente aparece **cinco
veces**, cada vez con el código completo de la versión anterior y la línea nueva marcada con `# NUEVO:`.

| versión | idea | qué agrega |
|---|---|---|
| v1 | `for` + `requests` | nada: línea base secuencial |
| v2 | hilos (`ThreadPoolExecutor`, 32) | concurrencia con un recurso (hilo) por llamada en vuelo |
| v3 | `gather` **sin tope** | concurrencia con corrutinas; se prueba sólo con 20 |
| v4 | `gather` + `Semaphore(32)` + `Limits(32)` | tope de entrada y tope de recursos |
| v5 | v4 + reintento + `asyncio.Queue` productor-consumidor | fallos transitorios y cola acotada |

**La cota.** Con $k$ llamadas en vuelo a la vez y $W=0.2$ s por llamada, $n$ llamadas no pueden tardar menos
que $\lceil n/k\rceil\cdot W$. Esa cota es la regla con la que se juzga cada versión.

Cargas de este tema: **a lo más 500 llamadas**. Antes de correr, el servidor de la clase debe estar prendido
(`FDD_SILENCIO=1 uv run python servidor.py`). Todo usa `await` de nivel superior: el kernel del notebook ya
tiene un event loop corriendo, y `asyncio.run` truena aquí.

In [ ]:
# Preparación del tema: imports, constantes y la tabla donde se acumulan las mediciones de las cinco versiones.
import asyncio, math, random, time
import httpx, requests
import matplotlib.pyplot as plt
from cable import URL
from gantt import registra, dibuja, reinicia

W = 0.2                                   # segundos que tarda cada llamada en el servidor (?lento=0.2)
LLAMADA = f"{URL}/ventas?lento=0.2"       # la llamada de todo el tema
RESULTADOS = {}                           # versión -> (n, k, segundos, nota)

def cota(n, k):
    """Lo mínimo que puede tardar n llamadas de W segundos con k en vuelo."""
    return math.ceil(n / k) * W

print("W =", W, "s · cota(500, 32) =", cota(500, 32), "s · cota(20, 1) =", cota(20, 1), "s")

## v1 · `for` + `requests`: la línea base

**Qué hace.** Una llamada, espera la respuesta, la siguiente. Un solo hilo, una sola llamada en vuelo
($k=1$). Es la versión que escribe cualquiera la primera vez.

**Por qué se mide con 20 y no con 500.** La cota es $\lceil n/1\rceil\cdot0.2$: 500 llamadas tardarían
**100 s**. Con 20 son 4 s, suficiente para ver la forma; el valor de 500 se calcula, y la tabla final lo
rotula como *estimado* para que nadie lo confunda con una medición.

**Qué esperar.** Un poco más de 4 s: 20 × 0.2 s más lo que cada llamada gasta en el cliente. El Gantt
muestra 20 barras de `espera` una detrás de otra, sin solaparse.

In [ ]:
# v1 · for + requests (secuencial). Repite 20 llamadas, una tras otra.
def lote_v1(n):
    marcas_t = []                                     # (inicio, fin) de cada llamada
    for i in range(n):
        ini = time.perf_counter()
        r = requests.get(LLAMADA, timeout=30)       # el hilo se bloquea aquí hasta la respuesta
        r.raise_for_status()
        marcas_t.append((ini, time.perf_counter()))
    return marcas_t

reinicia()
t0 = time.perf_counter()
marcas = lote_v1(20)
T = time.perf_counter() - t0
for i, (a, b) in enumerate(marcas):
    registra(f"llamada {i:02d}", a, b, "espera")   # una barra de espera por llamada
dibuja("v1 · for + requests · 20 llamadas")
RESULTADOS["v1"] = (20, 1, T, "medido con n=20; n=500 sería ≈ %.0f s (estimado)" % cota(500, 1))
print(f"v1: 20 llamadas en {T:.2f} s · cota {cota(20, 1):.2f} s · 500 llamadas (estimado): {cota(500, 1):.0f} s")
assert cota(20, 1) <= T < cota(20, 1) * 1.3, "v1 debería rondar la cota secuencial"

## v2 · hilos: `ThreadPoolExecutor(32)`

**Qué cambia.** La función de la llamada es **la misma** (`requests.get`). Lo nuevo es quién la ejecuta: un
pool de 32 hilos. Cada hilo se bloquea esperando su respuesta, pero hay 32 hilos esperando a la vez.

**Por qué 32.** Es el tope $k$ que se usa en toda la unidad. Con $k=32$ y $n=500$ la cota es
$\lceil500/32\rceil\cdot0.2=16\cdot0.2=3.2$ s.

**Qué esperar.** 20 llamadas ≈ 0.23 s (caben en una sola tanda), 100 ≈ 0.8 s y 500 ≈ 3.3 s: la forma de
escalera de la cota. Cada hilo cuesta memoria y cada llamada suelta de `requests.get` abre su propia
conexión; es la versión que "funciona" y la que más recursos del sistema gasta por llamada en vuelo.

In [ ]:
# v2 · hilos. Mismo código de v1 + el pool de hilos.
from concurrent.futures import ThreadPoolExecutor          # NUEVO: el pool de hilos

def una_llamada_v2(_):                                      # igual que en v1: bloquea al hilo que la corre
    r = requests.get(LLAMADA, timeout=30)
    r.raise_for_status()
    return r.status_code

def lote_v2(n, k=32):
    with ThreadPoolExecutor(max_workers=k) as pool:         # NUEVO: k hilos, cada uno hace llamadas
        return list(pool.map(una_llamada_v2, range(n)))     # NUEVO: map reparte las n entre los k hilos

for n in (20, 100, 500):
    ini = time.perf_counter()
    codigos = lote_v2(n)
    T = time.perf_counter() - ini
    RESULTADOS["v2" if n == 500 else f"v2 (n={n})"] = (n, 32, T, "hilos")
    print(f"v2: {n:>3} llamadas con 32 hilos: {T:5.2f} s · cota {cota(n, 32):.2f} s · {set(codigos)}")
assert RESULTADOS["v2"][2] < cota(500, 32) * 1.5

## v3 · `gather` **sin tope**, sólo con 20

**Qué cambia.** Se deja `requests` (bloqueante) por `httpx.AsyncClient`, cuyo `get` es una corrutina, y se
lanzan todas las tareas a la vez con `asyncio.gather`. No hay hilos: un solo hilo (el del loop) atiende
a todas, porque cada `await` le devuelve el hilo mientras la respuesta no llega.

**Por qué `return_exceptions=True`.** Sin esa opción, la primera excepción sale por el `await` de `gather`
y las demás tareas siguen corriendo sin que nadie recoja su resultado. Con la opción, cada excepción ocupa su
lugar en la lista de resultados y la versión puede contar fallos en lugar de abortar.

**Por qué sólo 20.** "Sin tope" aquí quiere decir *sin semáforo propio*: lo único que limita es el pool
por omisión de `httpx` (`max_connections=100`). Con 20 no se nota. Con 500, las 400 que no caben esperan
**dentro del pool**, y esa espera tiene dos costos medidos en esta máquina (`hechos_pool`, experimento h):
CPU del cliente (11.6 s de pared para 500 llamadas con 100 conexiones, contra 3.4 s con un semáforo de 32
por encima) y el `timeout` de 5 s por omisión, que **incluye** la espera por una conexión del pool. Por eso
la tarea de 500 se hace en v4, no aquí.

**Qué esperar.** 20 llamadas ≈ 0.23 s: la cota con $k=20$ es una sola tanda de 0.2 s.

In [ ]:
# v3 · gather sin tope. Mismo lote que v2, pero con corrutinas en lugar de hilos.
async def una_llamada_v3(cliente):                          # NUEVO: corrutina; el await cede el hilo
    r = await cliente.get(LLAMADA)                          # NUEVO: httpx asíncrono en lugar de requests
    r.raise_for_status()
    return r.status_code

async def lote_v3(n):
    async with httpx.AsyncClient(timeout=30) as cliente:   # NUEVO: un cliente (y un pool) para todas
        tareas = (una_llamada_v3(cliente) for _ in range(n))
        return await asyncio.gather(*tareas, return_exceptions=True)  # NUEVO: gather lanza las n a la vez

ini = time.perf_counter()
res = await lote_v3(20)
T = time.perf_counter() - ini
fallos = [r for r in res if isinstance(r, Exception)]       # las excepciones llegan como valores
RESULTADOS["v3"] = (20, 20, T, "gather sin tope, sólo n=20")
print(f"v3: 20 llamadas con gather sin tope: {T:.2f} s · cota {cota(20, 20):.2f} s · fallos: {len(fallos)}")
assert not fallos and T < 0.6

## v4 · `gather` + `Semaphore(32)` + `Limits(32)`

La página 4 corre esta versión en la celda 4.1. Aquí aparece **literal** (primer bloque) y después
ampliada en el bloque siguiente.

**Qué agrega.** Dos topes del mismo valor:

- `Semaphore(32)` es el tope de **entrada**: a lo más 32 tareas dentro del bloque `async with sem`. Las demás
  quedan bloqueadas en la cola del semáforo, que cuesta casi nada (una corrutina suspendida).
- `httpx.Limits(max_connections=32)` es el tope de **recursos**: a lo más 32 conexiones abiertas. Con el mismo
  valor, el pool nunca tiene cola propia.

**Qué esperar.** 20 → 0.23 s, 100 → 0.8 s, 500 → 3.4 s: cada tiempo se acerca a su cota por arriba.

In [ ]:
# celda 4.1
import asyncio, time, httpx
from cable import URL
async def pide(cliente, sem):
    async with sem:  # toma un permiso; si no hay, la tarea queda bloqueada en la cola
        return (await cliente.get(f"{URL}/ventas?lento=0.2")).status_code  # cede el hilo durante la espera

async def lote(n):
    sem = asyncio.Semaphore(32)  # 32 permisos: a lo más 32 dentro del bloque
    async with httpx.AsyncClient(limits=httpx.Limits(max_connections=32), timeout=30) as cliente:  # pool con el mismo tope
        return await asyncio.gather(*(pide(cliente, sem) for _ in range(n)))  # n tareas; resultados en orden

for n in (20, 100, 500):
    inicio = time.perf_counter()
    codigos = await lote(n)  # await de nivel superior: el notebook ya tiene un loop corriendo
    print(f"{n:>3} llamadas, asyncio con 32: {time.perf_counter() - inicio:5.2f} s  ({set(codigos)})")

### v4 ampliada: la misma celda, con medición de la espera por permiso y la curva $T$ contra $n$

**Qué se agrega a la celda 4.1.**

1. `return_exceptions=True`, como en v3.
2. Cada tarea anota **tres instantes**: cuando se creó, cuando obtuvo el permiso y cuando terminó. La
   diferencia entre el primero y el segundo es la espera por el permiso (`permiso` en el Gantt); la que hay entre
   el segundo y el tercero es la espera de la red (`espera`).
3. Un Gantt pequeño ($n=12$, $k=4$) para que se vean las tandas, y una curva con seis valores de $n$ contra
   la cota $\lceil n/k\rceil\cdot W$.

**Qué esperar en el Gantt.** Cuatro barras de `espera` por columna de tiempo; las demás tareas con una barra
ámbar (`permiso`) que termina justo cuando alguna otra libera su permiso.

In [ ]:
# v4 ampliada · semáforo + pool, medición de la espera por permiso y Gantt de una versión chica.
async def pide_v4(cliente, sem, i, t0, anota):
    creada = time.perf_counter()                             # la tarea existe pero no tiene permiso
    async with sem:                                          # entra: a lo más k dentro del bloque
        con_permiso = time.perf_counter()                    # NUEVO: el instante en que obtuvo el permiso
        r = await cliente.get(LLAMADA)                       # cede el hilo; el permiso sigue tomado
        r.raise_for_status()
        fin = time.perf_counter()
    if anota:                                                # NUEVO: anotar para el Gantt (sólo en lotes chicos)
        registra(f"tarea {i:02d}", creada, con_permiso, "permiso")
        registra(f"tarea {i:02d}", con_permiso, fin, "espera")
    return r.status_code

async def lote_v4(n, k=32, anota=False):
    sem = asyncio.Semaphore(k)                               # tope de entrada
    limites = httpx.Limits(max_connections=k)                # tope de recursos, el mismo valor
    t0 = time.perf_counter()
    async with httpx.AsyncClient(limits=limites, timeout=30) as cliente:
        res = await asyncio.gather(*(pide_v4(cliente, sem, i, t0, anota) for i in range(n)),
                                   return_exceptions=True)   # NUEVO: los fallos llegan como valores
    return res, time.perf_counter() - t0

reinicia()
res, T = await lote_v4(12, k=4, anota=True)                  # 12 tareas, 4 permisos: 3 tandas
dibuja("v4 · 12 llamadas con Semaphore(4) · la franja ámbar es la espera por el permiso")
print(f"12 llamadas con k=4: {T:.2f} s · cota {cota(12, 4):.2f} s")

puntos = []
for n in (20, 50, 100, 200, 350, 500):                       # NUEVO: la curva T contra n con k=32
    res, T = await lote_v4(n)
    fallos = sum(isinstance(r, Exception) for r in res)
    puntos.append((n, T, cota(n, 32), fallos))
    print(f"n={n:>3}  T={T:5.2f} s  cota={cota(n, 32):4.2f} s  T/cota={T / cota(n, 32):4.2f}  fallos={fallos}")
RESULTADOS["v4"] = (500, 32, puntos[-1][1], "Semaphore(32) + Limits(32)")

ns = list(range(1, 501))
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.step(ns, [cota(n, 32) for n in ns], where="post", color="#555555", label="cota ⌈n/32⌉·0.2 s")
ax.plot([p[0] for p in puntos], [p[1] for p in puntos], "o", color="#1f4e8c", label="medido (v4)")
ax.set_xlabel("llamadas n"); ax.set_ylabel("segundos"); ax.set_title("v4: el tiempo sigue la escalera de la cota")
ax.legend(); fig.tight_layout(); plt.show()
assert all(p[1] >= p[2] * 0.98 for p in puntos), "ninguna medición puede estar bajo la cota"
assert all(p[1] < p[2] * 1.6 for p in puntos), "v4 debería quedar cerca de la cota"
assert all(p[3] == 0 for p in puntos), "ninguna llamada debería fallar"

### La celda 4.2 (opcional): el orden y los fallos de `gather`

Dos reglas que sostienen a v3, v4 y v5:

1. `gather` devuelve los resultados **en el orden en que se pasaron las corrutinas**, no en el orden en que
   terminaron.
2. Con `return_exceptions=True`, una llamada que falla ocupa **su lugar** en la lista como una excepción. Esa
   segunda línea de la salida es la que imprime `HTTPStatusError`. Sin la opción, el `await` de `gather` relanzaría
   la primera excepción y el resto de la lista se perdería.

En la celda 4.2 la segunda ruta (`/ventas/99999`) no existe: el servidor responde 404.

In [ ]:
# celda 4.2
import asyncio, httpx
from cable import URL

async def pide(cliente, ruta):
    r = await cliente.get(f"{URL}{ruta}")
    r.raise_for_status()  # un 404 se vuelve excepción (httpx no lo hace solo)
    return r.status_code

rutas = ["/ventas?lento=0.3", "/ventas/99999?lento=0.1", "/ventas?lento=0.2"]
async with httpx.AsyncClient(timeout=30) as cliente:
    res = await asyncio.gather(*(pide(cliente, r) for r in rutas), return_exceptions=True)  # la excepción ocupa su lugar
for ruta, x in zip(rutas, res):
    print(f"{ruta:<24} {x if isinstance(x, int) else type(x).__name__}")

In [ ]:
# Comprobación de 4.2: el orden es el de las rutas aunque la segunda terminó primero (0.1 s) y la primera al final (0.3 s).
assert res[0] == 200 and isinstance(res[1], httpx.HTTPStatusError) and res[2] == 200

## v5 · v4 + reintento + `asyncio.Queue` productor-consumidor

Dos problemas que v4 no resuelve:

1. **Una falla transitoria arruina la llamada.** Si 1 de cada 25 llamadas falla una vez (red, reinicio de un
   proxy), v4 devuelve una excepción en esa posición y el trabajo pierde esa venta. v5 **reintenta** ante error
   hasta 3 veces, con espera creciente (*backoff*) y un poco de azar (*jitter*) para que los reintentos no
   lleguen todos juntos.
2. **`gather` crea las $n$ tareas de golpe.** Con 500 es barato; con 10 millones, no. v5 reparte el trabajo con
   una `asyncio.Queue(maxsize=…)`: **un productor** mete los ids en la cola (se suspende si está llena:
   contrapresión) y **$k$ consumidores** los sacan y llaman. Los consumidores son los $k$ recursos; ya no hace
   falta el `Semaphore`, porque sólo hay $k$ tareas que llaman.

**Cómo se prueba el reintento.** El servidor no falla solo. Se simula en el cliente y se rotula como tal:
toda llamada con `i % 25 == 0` lanza `httpx.ConnectError` en su **primer** intento. Son 20 de las 500; con el
reintento no debe quedar ningún fallo. Un reintento es seguro aquí porque un `GET` es idempotente; con un `POST`
que crea una venta, reintentar duplicaría el recurso.

In [ ]:
# v5 · v4 + reintento + cola productor-consumidor. Todo lo de v4 sigue aquí.
async def llama_v5(cliente, i, reintentos, intentos=3):
    for intento in range(1, intentos + 1):                   # NUEVO: hasta 3 intentos
        try:
            if i % 25 == 0 and intento == 1:                 # NUEVO: falla transitoria SIMULADA, sólo el 1.er intento
                raise httpx.ConnectError("falla simulada")
            r = await cliente.get(LLAMADA)                   # la llamada de v3 y v4
            r.raise_for_status()
            return r.status_code
        except httpx.HTTPError:
            if intento == intentos:
                raise                                        # NUEVO: se agotaron los intentos: la falla es real
            reintentos.append(i)                             # NUEVO: contar cuántos reintentos hubo
            await asyncio.sleep(0.05 * 2 ** intento + random.uniform(0, 0.05))  # NUEVO: backoff + jitter

async def lote_v5(n, k=32, tam_cola=64):
    cola = asyncio.Queue(maxsize=tam_cola)                   # NUEVO: cola acotada: el productor espera si se llena
    resultados, reintentos = {}, []

    async def productor():
        for i in range(n):
            await cola.put(i)                                # NUEVO: se suspende si la cola está llena
    async def consumidor(cliente):
        while True:                                          # NUEVO: cada consumidor es uno de los k recursos
            i = await cola.get()
            try:
                resultados[i] = await llama_v5(cliente, i, reintentos)
            except httpx.HTTPError as e:
                resultados[i] = e                            # se guarda el fallo, no se pierde el consumidor
            finally:
                cola.task_done()

    limites = httpx.Limits(max_connections=k)
    async with httpx.AsyncClient(limits=limites, timeout=30) as cliente:
        consumidores = [asyncio.create_task(consumidor(cliente)) for _ in range(k)]   # NUEVO: k consumidores
        await productor()
        await cola.join()                                    # NUEVO: espera a que se procese todo lo encolado
        for c in consumidores:
            c.cancel()                                       # NUEVO: ya no hay más trabajo
        await asyncio.gather(*consumidores, return_exceptions=True)
    return resultados, reintentos

t0 = time.perf_counter()
resultados, reintentos = await lote_v5(500)
T = time.perf_counter() - t0
fallos = [i for i, r in resultados.items() if isinstance(r, Exception)]
RESULTADOS["v5"] = (500, 32, T, f"{len(reintentos)} reintentos, {len(fallos)} fallos")
print(f"v5: 500 llamadas con 32 consumidores: {T:.2f} s · cota {cota(500, 32):.2f} s")
print(f"reintentos: {len(reintentos)} (esperados 20) · fallos finales: {len(fallos)} · resultados: {len(resultados)}")
assert len(resultados) == 500 and not fallos
assert len(reintentos) == 20, "20 llamadas (i múltiplo de 25) fallan una vez y se reintentan"
assert cota(500, 32) * 0.98 <= T < cota(500, 32) * 1.6

## La tabla final: v1 … v5

Todas las cifras salen de las celdas de arriba, en **esta** máquina y en **esta** corrida. La columna
*cota* es $\lceil n/k\rceil\cdot W$ con el $k$ de la versión. v1 se midió con $n=20$; el valor para $n=500$ es
la cota, no una medición.

**Cómo leerla.**

- v1 → v2 cambia **quién espera** (un hilo por llamada en vuelo) y gana ×30 en tiempo.
- v2 → v3 cambia **el costo de esperar** (una corrutina en lugar de un hilo) pero no el tiempo: la cota es la
  misma. Lo que se gana es poder subir $n$ sin crear hilos.
- v3 → v4 agrega **el tope**: sin él, 500 llamadas pelean por el pool de `httpx`.
- v4 → v5 agrega **tolerancia a fallos** y una cola acotada. El tiempo no mejora; la corrección sí.

In [ ]:
print(f"{'versión':<11}{'n':>5}{'k':>5}{'segundos':>10}{'cota':>8}  nota")
for nombre in ("v1", "v2", "v3", "v4", "v5"):
    n, k, T, nota = RESULTADOS[nombre]
    print(f"{nombre:<11}{n:>5}{k:>5}{T:>10.2f}{cota(n, k):>8.2f}  {nota}")
assert set(RESULTADOS) >= {"v1", "v2", "v3", "v4", "v5"}
# v1 con 500 llamadas (estimado) contra v4: cuántas veces más rápida es la versión acotada
print(f"v4 contra v1 (v1 con n=500 estimado): ×{cota(500, 1) / RESULTADOS['v4'][2]:.0f}")
assert RESULTADOS["v4"][2] < cota(500, 1) / 20

# Tema 5 · Pools y recursos: quién corre, quién entra, qué conexión usas

**Pregunta.** Un pool tiene 5 conexiones y las 5 están en vuelo, esperando respuesta. ¿Puede la sexta petición
usar una de ellas? ¿Se reutiliza una conexión cuando su petición termina? ¿Qué libera `await`?

Este tema es la versión verbosa de la página 5. En lugar de afirmar las respuestas, **las cuenta**: el servidor
de la clase tiene un endpoint `GET /quien` que responde tres datos de **tu conexión**:

| campo | qué es | para qué sirve aquí |
|---|---|---|
| `puerto` | el puerto del cliente de esa conexión TCP | un puerto distinto = una conexión distinta |
| `hilo` | el hilo del servidor que la atiende | conexiones abiertas del cliente = hilos ocupados del servidor |
| `peticion_en_conexion` | cuántas peticiones lleva esa conexión | si crece (1, 2, 3, …) la conexión se reutilizó |

**Tres topes que se confunden** (y que este tema separa con mediciones):

| tope | quién lo pone | qué cuenta | qué guarda |
|---|---|---|---|
| de tareas | nadie (`gather` sin límite) | corrutinas creadas | nada: son baratas |
| de entrada | `Semaphore(k)` | permisos | sólo un número |
| de recursos | el pool (`httpx.Limits`, una `Queue` de conexiones) | conexiones reales | **los objetos**: los presta, los recupera, los reutiliza |

Todo el tema usa `await` de nivel superior. El servidor de la clase debe estar prendido
(`FDD_SILENCIO=1 uv run python servidor.py`).

In [ ]:
# Preparación del tema: imports y un diccionario donde se acumulan las cifras que el cierre comprueba.
import asyncio, logging, statistics, threading, time
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
import httpx, requests
from requests.adapters import HTTPAdapter
from cable import URL
from gantt import registra, dibuja, reinicia

CIFRAS = {}          # nombre -> valor medido; el cierre del tema las comprueba y las imprime
print("listo · URL =", URL)

## 5.1 · ¿Cuántas conexiones abre cada cliente?

`requests.get(url)` crea una `Session`, la usa **una vez** y la tira: cada llamada es una conexión TCP nueva
(un puerto nuevo, un hilo nuevo en el servidor). `requests.Session()` guarda la conexión abierta en su pool y
la reutiliza: un solo puerto, y `peticion_en_conexion` sube 1, 2, 3, …

**Qué esperar.** Suelto: 20 puertos, 20 hilos del servidor, 1 petición por conexión. `Session`: 1 puerto,
1 hilo, hasta 20 peticiones en esa conexión.

In [ ]:
# celda 5.1
import requests
from cable import URL

def resume(nombre, vistos):                   # vistos: las respuestas de /quien
    puertos = {v["puerto"] for v in vistos}   # puerto del cliente: uno por conexión TCP
    hilos = {v["hilo"] for v in vistos}       # hilo del servidor que atendió esa conexión
    n_max = max(v["peticion_en_conexion"] for v in vistos)  # peticiones de la conexión más usada
    print(f"{nombre}: {len(puertos)} puertos · {len(hilos)} hilos del servidor"
          f" · hasta {n_max} peticiones por conexión")

resume("suelto ", [requests.get(f"{URL}/quien").json() for _ in range(20)])  # una conexión por llamada
with requests.Session() as s:                 # la Session guarda la conexión entre llamadas
    resume("Session", [s.get(f"{URL}/quien").json() for _ in range(20)])

### 5.1 ampliada: los tres clientes lado a lado, y cuánto ahorra reutilizar

Se agrega el tercer cliente, `httpx.AsyncClient(limits=httpx.Limits(max_connections=5))`, con 20 peticiones
**simultáneas** de 0.2 s. Con `requests` las peticiones eran en serie, por eso una sola conexión bastaba; aquí
hay 20 en vuelo y 5 conexiones, y la pregunta es cuántos puertos distintos ve el servidor.

También se mide cuánto cuesta no reutilizar: 200 llamadas rápidas (`lento=0`) sin y con `Session`.

**Qué esperar.** `httpx` con `Limits(5)`: **a lo más 5 puertos**. La diferencia por llamada entre suelto y
`Session` es pequeña en localhost (décimas de milisegundo): el handshake TCP local casi no cuesta. El ahorro
que se cita en la literatura (TCP + TLS + autenticación, uno a tres viajes de ida y vuelta por conexión) es de
**red real**; en esta máquina no se ve, y por eso la cifra no se presenta como si lo fuera.

In [ ]:
# Los tres clientes con el mismo resumen, y el costo de no reutilizar.
async def httpx_con_limite(k, n=20, lento=0.2):
    async with httpx.AsyncClient(limits=httpx.Limits(max_connections=k), timeout=30) as c:
        pedir = lambda: c.get(f"{URL}/quien?lento={lento}")             # NUEVO: peticiones simultáneas
        resp = await asyncio.gather(*(pedir() for _ in range(n)))
    return [r.json() for r in resp]

t0 = time.perf_counter()
vistos_httpx = await httpx_con_limite(5)                                # NUEVO: 20 en vuelo, pool de 5
T_httpx = time.perf_counter() - t0
resume("httpx  ", vistos_httpx)
print(f"       20 peticiones de 0.2 s con Limits(5): {T_httpx:.2f} s")

def mide_200(llamar):
    t0 = time.perf_counter()
    for _ in range(200):
        llamar(f"{URL}/quien?lento=0")                                  # 200 llamadas que responden de inmediato
    return time.perf_counter() - t0

suelto = mide_200(requests.get)
with requests.Session() as s:
    sesion = mide_200(s.get)
print(f"200 llamadas sin Session: {suelto:.2f} s  ({suelto / 200 * 1000:.2f} ms c/u)")
print(f"200 llamadas con Session: {sesion:.2f} s  ({sesion / 200 * 1000:.2f} ms c/u)")
print(f"diferencia por llamada:   {(suelto - sesion) / 200 * 1000:.2f} ms  (localhost: sin TLS ni viaje de red)")

puertos_suelto = len({requests.get(f"{URL}/quien").json()["puerto"] for _ in range(20)})
with requests.Session() as s:
    puertos_sesion = len({s.get(f"{URL}/quien").json()["puerto"] for _ in range(20)})
CIFRAS.update(puertos_suelto=puertos_suelto, puertos_sesion=puertos_sesion,
              puertos_httpx5=len({v["puerto"] for v in vistos_httpx}), t_httpx5=T_httpx)
assert puertos_suelto == 20 and puertos_sesion == 1
assert CIFRAS["puertos_httpx5"] <= 5

## 5.2 · Un pool de 5 conexiones y 20 peticiones

Las 20 tareas existen a la vez (tope de tareas: 20), pero sólo hay 5 conexiones (tope de recursos: 5). Las 15
tareas restantes **esperan en la cola del pool** y entran en cuanto una conexión vuelve: salen en olas de 5.
Cada ola dura 0.2 s, así que 4 olas ≈ 0.8 s.

La sexta petición **no puede** usar ninguna de las cinco conexiones mientras sigan en vuelo: en HTTP/1.1 una
conexión lleva una petición a la vez. Cuando la petición de una conexión termina, la conexión **sí se reutiliza**
(`keep-alive`): la sexta la toma sin abrir otra.

**Qué esperar.** 0.8 s, 5 puertos, 5 peticiones por ola, y 4 peticiones por cada conexión.

In [ ]:
# celda 5.2
import asyncio, time
from collections import Counter
import httpx

limites = httpx.Limits(max_connections=5)     # el pool del cliente: a lo más 5 conexiones
async with httpx.AsyncClient(limits=limites, timeout=30) as cliente:
    async def una(i):
        d = (await cliente.get(f"{URL}/quien?lento=0.2")).json()   # 0.2 s en el servidor
        return i, round((time.perf_counter() - t0) / 0.2), d       # ola = fin / 0.2 s
    t0 = time.perf_counter()
    res = await asyncio.gather(*(una(i) for i in range(20)))      # 20 a la vez
print(f"20 peticiones, pool de 5: {time.perf_counter() - t0:.2f} s")
print("puertos distintos:", len({d["puerto"] for _, _, d in res}))
print("peticiones por ola:", dict(sorted(Counter(ola for _, ola, _ in res).items())))
print("peticiones por conexión:", sorted(Counter(d["puerto"] for _, _, d in res).values()))

### 5.2 ampliada: el Gantt por petición

Se repite el experimento anotando, para cada petición, **dos** intervalos:

- `permiso` (ámbar): desde que se creó la tarea hasta que consiguió una conexión. Es la espera **en la cola del
  pool**.
- `espera` (contorno): la petición en vuelo, 0.2 s en el servidor.

Un detalle de honestidad: `httpx` no avisa cuándo una petición obtiene la conexión. Aquí el inicio del vuelo se
**estima** como `fin − 0.2 s` (el servidor duerme exactamente 0.2 s); el fin sí es medido. Con tareas cuyo
tiempo en el servidor se conoce eso basta para que el dibujo muestre las olas.

**Qué esperar.** Cinco filas sin barra ámbar (ola 1), cinco con 0.2 s de ámbar (ola 2), y así hasta 0.6 s.

In [ ]:
# Gantt del pool de 5: una fila por petición, ámbar = esperó en la cola del pool.
async def una_g(cliente, i, creada):
    d = (await cliente.get(f"{URL}/quien?lento=0.2")).json()
    fin = time.perf_counter()
    ini = fin - 0.2                                            # NUEVO: inicio del vuelo estimado
    if ini - creada > 0.02:
        registra(f"petición {i:02d}", creada, ini, "permiso")  # NUEVO: esperó en la cola del pool
    registra(f"petición {i:02d}", max(ini, creada), fin, "espera")
    return d

reinicia()
async with httpx.AsyncClient(limits=httpx.Limits(max_connections=5), timeout=30) as cliente:
    creada = time.perf_counter()
    vistos = await asyncio.gather(*(una_g(cliente, i, creada) for i in range(20)))
T_pool5 = time.perf_counter() - creada
dibuja("Pool de 5 conexiones, 20 peticiones de 0.2 s · ámbar = espera en la cola del pool")
por_conexion = sorted(Counter(d["puerto"] for d in vistos).values())
print(f"{T_pool5:.2f} s · puertos {len(por_conexion)} · peticiones por conexión {por_conexion}")
CIFRAS.update(pool5_20_s=T_pool5, pool5_puertos=len(por_conexion))
assert 0.75 <= T_pool5 < 1.3 and len(por_conexion) <= 5
assert sum(por_conexion) == 20 and max(por_conexion) >= 4, "las conexiones se reutilizan"

## 5.3 · Un pool es una cola de objetos

Un pool de conexiones es, en esencia, una **cola de objetos reales** con dos operaciones: *prestar* (sacar uno;
si no queda ninguno, esperar) y *devolver* (volver a meterlo). Un `Semaphore` sólo cuenta; no sabe qué objetos hay.
La clase siguiente es el pool completo, sin abrir ni cerrar conexiones de verdad (son cadenas de texto que
representan conexiones a una BD).

Con ella se mide la regla de la página: **suelta el recurso antes de la espera larga**. Una tarea usa la
conexión 0.2 s (consulta) y luego espera 0.5 s a *otra* API que no necesita la BD. Con $k=3$ conexiones y 12
tareas:

- *retener* hasta el final: cada tarea ocupa la conexión 0.7 s → $4\times0.7=2.8$ s;
- *soltar* antes de la espera larga: cada tarea la ocupa 0.2 s → $4\times0.2+0.5=1.3$ s.

**Qué esperar.** 2.8 s contra 1.3 s: la misma BD, la misma carga, más del doble de capacidad por haber soltado
antes.

In [ ]:
# La clase Pool: una cola de conexiones. prestar = get, devolver = put.
class Pool:
    def __init__(self, k):
        self.libres = asyncio.Queue()                  # la cola de objetos reales
        for i in range(k):
            self.libres.put_nowait(f"conexión-{i}")    # k conexiones (simuladas)
        self.k = k
    async def prestar(self):
        return await self.libres.get()                 # si no queda ninguna, la tarea se suspende aquí
    def devolver(self, conexion):
        self.libres.put_nowait(conexion)               # otra tarea suspendida la recibe
    def en_uso(self):
        return self.k - self.libres.qsize()

p = Pool(2)
a = await p.prestar(); b = await p.prestar()
print("prestadas:", a, b, "· en uso:", p.en_uso(), "· libres:", p.libres.qsize())
try:
    await asyncio.wait_for(p.prestar(), timeout=0.1)   # la tercera no puede: no hay conexión libre
except asyncio.TimeoutError:
    print("la tercera petición esperó 0.1 s y no consiguió conexión: el pool está lleno")
p.devolver(a)
print("devuelta", a, "· ahora la toma la siguiente:", await p.prestar())
assert p.en_uso() == 2

In [ ]:
# celda 5.3
async def con_pool(retener, k=3, n=12):
    pool = asyncio.Queue()
    for i in range(k):
        pool.put_nowait(f"conexión-{i}")      # k conexiones (simuladas) a la BD
    async def tarea():
        c = await pool.get()                  # pedir prestada: espera si no queda ninguna
        await asyncio.sleep(0.2)              # la consulta usa la conexión
        if not retener: pool.put_nowait(c)    # soltar ANTES de la espera larga
        await asyncio.sleep(0.5)              # otra API: no necesita la BD
        if retener: pool.put_nowait(c)        # soltar DESPUÉS: la tuvo 0.7 s
    t0 = time.perf_counter()
    await asyncio.gather(*(tarea() for _ in range(n)))
    return time.perf_counter() - t0
for retener in (True, False):
    print(f"{'retiene' if retener else 'suelta '} la conexión: {await con_pool(retener):.2f} s")

### 5.3 ampliada: lo mismo con la clase `Pool` y con Gantt

Misma comparación, ahora con la clase `Pool` y con 6 tareas para que el dibujo quepa. En el Gantt, ámbar es
la espera por una conexión; el contorno es "la tarea usa la conexión" (consulta) o "la tarea espera a la otra
API". Observa **cuándo** reaparece el ámbar en el segundo dibujo: casi nunca, porque las conexiones vuelven al
pool a los 0.2 s.

In [ ]:
# Retener contra soltar, con la clase Pool y con Gantt (6 tareas, k=3).
async def tarea_pool(pool, i, retener):
    creada = time.perf_counter()
    c = await pool.prestar()                                   # NUEVO: el pool, no la cola suelta
    obtiene = time.perf_counter()
    await asyncio.sleep(0.2)                                   # consulta: la conexión está en uso
    if not retener: pool.devolver(c)                           # suelta ANTES de la espera larga
    consulta = time.perf_counter()
    await asyncio.sleep(0.5)                                   # otra API
    fin = time.perf_counter()
    if retener: pool.devolver(c)
    registra(f"tarea {i}", creada, obtiene, "permiso")         # NUEVO: espera por la conexión
    registra(f"tarea {i}", obtiene, consulta, "espera")        # consulta en la BD
    registra(f"tarea {i}", consulta, fin, "espera")            # otra API

tiempos = {}
for retener in (True, False):
    reinicia()
    pool = Pool(3)
    t0 = time.perf_counter()
    await asyncio.gather(*(tarea_pool(pool, i, retener) for i in range(6)))
    tiempos[retener] = time.perf_counter() - t0
    dibuja(f"{'RETIENE' if retener else 'SUELTA'} la conexión · 6 tareas, k=3 · {tiempos[retener]:.2f} s")
retiene_s, suelta_s = await con_pool(True), await con_pool(False)   # los números de las 12 tareas, como en 5.3
CIFRAS.update(retiene_s=retiene_s, suelta_s=suelta_s)
print(f"12 tareas, k=3: retiene {retiene_s:.2f} s · suelta {suelta_s:.2f} s · razón {retiene_s / suelta_s:.2f}")
assert 2.7 <= retiene_s < 3.2 and 1.25 <= suelta_s < 1.6
assert tiempos[True] > tiempos[False]

## 5.4 · Semáforo y pool juntos: ¿quién limita?

Cuando hay un `Semaphore(ks)` por encima de un pool de `kp` conexiones, lo que realmente está en vuelo es
$\min(k_s,k_p)$. Tres combinaciones con 20 peticiones de 0.2 s:

| `Semaphore` | pool | quien manda | olas $\lceil20/\min\rceil$ | tiempo esperado |
|---:|---:|---|---:|---:|
| 32 | 5 | pool | 4 | 0.8 s |
| 5 | 5 | los dos | 4 | 0.8 s |
| 3 | 5 | semáforo | 7 | 1.4 s |

Con `Semaphore(3)` el pool **nunca abre más de 3 conexiones**: no hay petición que las pida. El puerto es la
prueba: mide cuántas conexiones se abrieron de verdad.

In [ ]:
# celda 5.4
async def lote(permisos, conexiones, n=20):
    sem = asyncio.Semaphore(permisos)                       # tope de entrada: cuenta permisos
    limites = httpx.Limits(max_connections=conexiones)      # tope de recursos: conexiones reales
    async with httpx.AsyncClient(limits=limites, timeout=30) as c:
        async def una():
            async with sem:                                 # el permiso se retiene durante el await
                return (await c.get(f"{URL}/quien?lento=0.2")).json()["puerto"]
        t0 = time.perf_counter()
        puertos = await asyncio.gather(*(una() for _ in range(n)))
    return time.perf_counter() - t0, len(set(puertos))

for permisos, conexiones in [(32, 5), (5, 5), (3, 5)]:
    t, p = await lote(permisos, conexiones)
    print(f"Semaphore({permisos:2d}) + pool de {conexiones}: {t:.2f} s · {p} puertos")

### 5.4 ampliada: la tabla contra la predicción

Se calcula la predicción $\lceil n/\min(k_s,k_p)\rceil\cdot0.2$ s y el número de puertos esperado
($\min(k_s,k_p)$) y se compara con cada medición. Dos combinaciones más: `Semaphore(2)` y `Semaphore(1)`.

In [ ]:
# Tabla: predicción contra medición para más combinaciones.
import math
filas = []
for permisos, conexiones in [(32, 5), (5, 5), (3, 5), (2, 5), (1, 5)]:
    t, p = await lote(permisos, conexiones)
    esperado = math.ceil(20 / min(permisos, conexiones)) * 0.2
    filas.append((permisos, conexiones, t, esperado, p, min(permisos, conexiones)))
print(f"{'sem':>4}{'pool':>5}{'medido':>8}{'predicho':>9}{'puertos':>8}{'esperados':>10}")
for f in filas:
    print(f"{f[0]:>4}{f[1]:>5}{f[2]:>8.2f}{f[3]:>9.2f}{f[4]:>8}{f[5]:>10}")
CIFRAS["sem3_pool5_s"] = filas[2][2]
assert all(f[2] >= f[3] * 0.98 for f in filas), "ninguna medición puede estar bajo la predicción"
assert all(f[2] < f[3] * 1.5 for f in filas)
assert all(f[4] <= f[5] for f in filas), "nunca más conexiones que min(sem, pool)"

## 5.5 (ejercicio) · 500 llamadas con un pool de 100: con y sin `Semaphore(32)`

Con 500 llamadas de 0.2 s y 100 conexiones, lo ideal son 5 olas ≈ 1 s. Pero con **100 permisos** (o sin
semáforo), la medición en esta máquina tarda **varios segundos** y abre más puertos que conexiones permitidas. La
causa se midió en `.superpowers/medidas-10-2/hechos_pool.md`: con un pool de 100 y 500 peticiones, la cola
vive **dentro** de `httpx` y su gestión recorre cola × conexiones en cada evento. Es **CPU del cliente**, no
espera de red. Con un semáforo de 32 **por encima**, el pool sólo ve 32 peticiones a la vez y la cola queda en
el semáforo, que es barata.

La celda 5.5 de la página mide una vez cada combinación. Después, en la ampliación, el ejercicio es **repetirla
3 veces** y mirar la mediana y la CPU del cliente (`time.process_time()` mide el tiempo de CPU del proceso, no
el reloj de pared).

**Qué esperar.** Con 100 permisos: ≈ 8–10 s de pared, casi todos de CPU. Con 32: ≈ 3.4 s. Esta corrida tarda
≈ 45 s en total.

In [ ]:
# celda 5.5
for permisos, conexiones in [(100, 100), (32, 100)]:
    t, p = await lote(permisos, conexiones, n=500)          # la función de la celda 5.4
    print(f"500 llamadas, Semaphore({permisos}) + pool de {conexiones}: {t:.2f} s · {p} puertos")

### 5.5 ampliada: tres repeticiones, mediana y CPU del cliente

Una sola medición engaña. Se repite tres veces cada combinación y se reporta la mediana. La columna *CPU* es
`time.process_time()` antes y después: si la CPU del cliente es casi igual al tiempo de pared, el cliente
estuvo ocupado calculando, no esperando.

In [ ]:
# Ejercicio: 3 repeticiones por combinación, con mediana y CPU del cliente.
REPETICIONES = 3
tabla = {}
for permisos, conexiones in [(100, 100), (32, 100)]:
    corridas = []
    for _ in range(REPETICIONES):
        cpu0 = time.process_time()                          # NUEVO: tiempo de CPU del proceso
        t, p = await lote(permisos, conexiones, n=500)
        corridas.append((t, time.process_time() - cpu0, p))
    tabla[permisos] = corridas
print(f"{'semáforo':>9}{'pool':>6}{'mediana s':>10}{'CPU s':>7}{'puertos':>8}   todas")
for permisos, corridas in tabla.items():
    mediana = statistics.median(c[0] for c in corridas)
    cpu = statistics.median(c[1] for c in corridas)
    puertos = max(c[2] for c in corridas)
    todas = " ".join(f"{c[0]:.1f}" for c in corridas)
    print(f"{permisos:>9}{100:>6}{mediana:>10.2f}{cpu:>7.2f}{puertos:>8}   {todas}")
med100 = statistics.median(c[0] for c in tabla[100])
med32 = statistics.median(c[0] for c in tabla[32])
CIFRAS.update(pool100_500_s=med100, pool100_sem32_500_s=med32)
print(f"la cola dentro del pool cuesta ×{med100 / med32:.1f} el tiempo del semáforo de 32 por encima")
assert med32 < med100, "con el semáforo de 32 por encima, el lote debe ser más rápido"
assert med32 < 5.0

## 5.6 · Un pool que no limita: `requests.Session` con 20 hilos

`requests` también tiene un pool (el de `urllib3`, con `pool_maxsize=10` por omisión), y es la fuente de una
confusión frecuente: **ese pool no hace esperar a nadie**. Es un *caché* de conexiones, no un tope. Con
`pool_block=False` (el valor por omisión), si 20 hilos piden conexión a la vez, `urllib3` abre **20**; cuando
vuelven, guarda 10 y **tira 10** con el aviso `Connection pool is full, discarding connection`.

Para ver el aviso se engancha un *handler* al logger `urllib3.connectionpool`. Después se repite con
`HTTPAdapter(pool_maxsize=5, pool_block=True)`: ahora el pool sí es un tope y hace esperar.

**Qué esperar.** Primera parte: 20 puertos y ≈ 10 avisos, en ≈ 0.22 s. Segunda: ≤ 5 puertos y ≈ 0.8 s.

In [ ]:
# Session compartida por 20 hilos: el pool NO limita (pool_block=False) y tira conexiones.
avisos = []
class Captura(logging.Handler):
    def emit(self, record):
        avisos.append(record.getMessage())                      # guarda el texto de cada aviso
logger = logging.getLogger("urllib3.connectionpool")
manejador = Captura(level=logging.WARNING)
logger.addHandler(manejador)

def pedir_quien(s, _):
    return s.get(f"{URL}/quien?lento=0.2").json()               # cada hilo hace una petición de 0.2 s

with requests.Session() as s, ThreadPoolExecutor(max_workers=20) as hilos:
    t0 = time.perf_counter()
    vistos = list(hilos.map(lambda i: pedir_quien(s, i), range(20)))   # 20 hilos, 20 peticiones a la vez
    T_sin_bloqueo = time.perf_counter() - t0
puertos_sin = len({v["puerto"] for v in vistos})
print(f"Session, 20 hilos, pool_maxsize=10 (por omisión): {T_sin_bloqueo:.2f} s · {puertos_sin} puertos · {len(avisos)} avisos")
print("ejemplo de aviso:", avisos[0] if avisos else "(ninguno)")
n_avisos = len(avisos)                                          # se guarda antes de limpiar la lista

# NUEVO: el mismo experimento con pool_block=True y pool_maxsize=5
avisos.clear()
with requests.Session() as s, ThreadPoolExecutor(max_workers=20) as hilos:
    s.mount("http://", HTTPAdapter(pool_connections=1, pool_maxsize=5, pool_block=True))   # NUEVO: el pool hace esperar
    t0 = time.perf_counter()
    vistos = list(hilos.map(lambda i: pedir_quien(s, i), range(20)))
    T_con_bloqueo = time.perf_counter() - t0
puertos_con = len({v["puerto"] for v in vistos})
print(f"Session, 20 hilos, pool_maxsize=5, pool_block=True: {T_con_bloqueo:.2f} s · {puertos_con} puertos · {len(avisos)} avisos")
logger.removeHandler(manejador)
CIFRAS.update(sesion20_puertos=puertos_sin, sesion20_avisos=n_avisos)
assert n_avisos >= 1, "debió salir al menos un aviso Connection pool is full"
assert puertos_sin > 10, "sin pool_block el pool de requests abre más conexiones que su maxsize"
assert puertos_con <= 5 and T_con_bloqueo >= 0.75
assert T_con_bloqueo > T_sin_bloqueo * 2

## Cierre del tema: la tabla de lo medido

Cada renglón responde una de las preguntas de la página y viene de una celda de arriba.

| pregunta | respuesta medida |
|---|---|
| ¿`requests.get` suelto reutiliza la conexión? | no: un puerto por llamada |
| ¿`Session` la reutiliza? | sí: 1 puerto, `peticion_en_conexion` crece |
| ¿el pool de 5 deja que la 6.ª use una de las 5 en vuelo? | no: espera en la cola del pool |
| ¿se reutiliza la conexión que terminó? | sí: 4 peticiones por conexión |
| ¿retener o soltar? | soltar antes de la espera larga: la misma BD rinde más del doble |
| ¿qué limita, el semáforo o el pool? | el menor de los dos |
| ¿cuándo cuesta CPU la cola del pool? | con muchas peticiones en la cola de `httpx` |
| ¿el pool de `requests` limita? | no, salvo `pool_block=True` |

In [ ]:
print(f"{'cifra':<24}{'valor':>10}")
for nombre, valor in CIFRAS.items():
    print(f"{nombre:<24}{valor:>10.2f}" if isinstance(valor, float) else f"{nombre:<24}{valor:>10}")
assert CIFRAS["puertos_suelto"] == 20 and CIFRAS["puertos_sesion"] == 1
assert CIFRAS["pool5_puertos"] <= 5 and CIFRAS["retiene_s"] > 2 * CIFRAS["suelta_s"] * 0.95
assert CIFRAS["pool100_sem32_500_s"] < CIFRAS["pool100_500_s"]
print("tema 5: todas las comprobaciones pasan")

# Tema 6 · Calcular en paralelo: el loop bloqueado, los procesos y la curva de Amdahl

**Pregunta.** Dentro de un servicio `async` hay un cálculo de 2 s en Python puro. ¿Qué le pasa a las otras
tareas? ¿Se arregla con un hilo? ¿Con un proceso? ¿Cuánto se gana con 1, 2 y 4 procesos, y por qué no se gana
lo que dice el número de procesos?

Este tema es la versión verbosa de la página 6. Todo el cálculo es **local**: no necesita el servidor de la
clase. `calcula` vive en `trabajo_computo.py` y no en una celda, porque un proceso hijo recibe de `pickle` el
*nombre* de la función (módulo + nombre) y la busca importando ese módulo; una función definida en la celda
pertenece a `__main__` del kernel, que el hijo no puede importar.

**Lo que se mide en este tema**

| medición | celda | qué prueba |
|---|---|---|
| latido con el cálculo **en** el loop | 6.1 | un cálculo sin `await` detiene a todas las tareas del loop |
| latido con el cálculo en un **hilo** | 6.1 ampliada | el loop no se detiene, pero el cálculo no se acelera (GIL) |
| latido con el cálculo en un **proceso** | 6.2 | el loop sigue y el cálculo corre en otra CPU |
| 4 tareas con 1, 2 y 4 procesos | 6.3 ampliada | speedup, eficiencia y la fracción serial $f$ |
| una `lambda` a un proceso | cierre | `pickle` manda funciones por nombre |

Cargas del tema: **a lo más 4 procesos** y cálculos de 0.3 a 2 s.

In [ ]:
# Preparación del tema: imports y la tabla donde se acumulan las cifras.
import asyncio, math, pickle, time
from concurrent.futures import ProcessPoolExecutor
import matplotlib.pyplot as plt
from trabajo_computo import calcula, calcula_medido
from gantt import registra, dibuja, reinicia

CIFRAS = {}
print("calcula(1000) =", calcula(1000), "· determinista: mismo n, mismo resultado")

## 6.1 · Un latido y un cálculo en el mismo loop

`latido` imprime una línea cada 0.1 s con `await asyncio.sleep(0.1)`: cada `await` cede el hilo. `calculo`
espera 0.15 s (para que salgan dos latidos) y después llama a `calcula()`, **sin ningún `await`**. Mientras
`calcula()` ejecuta, el hilo del loop está ocupado y *nadie más corre*. Cuando `latido` despierta de su `sleep`,
queda en estado `lista`: puede ejecutar, pero no tiene el hilo.

**Qué esperar.** Los latidos 0 y 1 salen a 0.00 y 0.10 s; el cálculo dura entre 1 y 2 s según la CPU; el latido 2 sale con ese mismo
retraso en lugar de 0.1 s. No hay error ni aviso: un loop bloqueado sólo se nota en la latencia.

In [ ]:
# celda 6.1
import asyncio
import time
from trabajo_computo import calcula

async def latido(veces):
    for i in range(veces):
        print(f"{time.perf_counter() - t0:4.2f} s  latido {i}")
        await asyncio.sleep(0.1)                   # cede el loop 0.1 s
async def calculo():
    await asyncio.sleep(0.15)                      # deja salir dos latidos antes
    inicio = time.perf_counter()
    calcula()                                      # ~2 s de Python sin ningún await
    print(f"      el cálculo duró {time.perf_counter() - inicio:.2f} s sin ceder el hilo")
t0 = time.perf_counter()
await asyncio.gather(latido(5), calculo())         # las dos, en el mismo loop

### 6.1 ampliada: tres dónde-corre-el-cálculo, con la misma medición

El latido de la celda 6.1 imprime; para **comparar** hace falta un número. `latido_g` anota el instante de
cada latido y el Gantt de lo que pasó: el latido es una fila con tres tipos de barra —`espera` (el `sleep`),
`lista` (el `sleep` ya venció pero no hay hilo) y una marca `exec` muy corta cuando imprime—. El número que
resume la corrida es el **mayor intervalo entre latidos consecutivos**: ideal, 0.1 s.

Se comparan tres sitios para el mismo `calcula()`:

| sitio | cómo | qué se espera |
|---|---|---|
| en el loop | `calcula()` directo | intervalo máximo ≈ duración del cálculo |
| en un hilo | `await asyncio.to_thread(calcula)` | el loop no se detiene (el intérprete cede el GIL cada ≈ 5 ms), pero el cálculo y el latido **comparten un GIL** |
| en un proceso | `await loop.run_in_executor(pool, calcula)` | el loop no nota nada; el cálculo corre en otra CPU con otro GIL |

**Qué esperar.** Intervalo máximo ≈ la duración del cálculo en el loop, ≈ 0.1 s en el hilo y en el proceso. La diferencia entre
hilo y proceso no se ve en el latido sino en el **cálculo**: en un hilo no se acelera ni se reparte entre CPU.

In [ ]:
# Misma medición para tres sitios. NUEVO: latido_g anota cada latido y el Gantt.
async def latido_g(duracion, marcas):
    inicio = time.perf_counter()
    while time.perf_counter() - inicio < duracion:
        antes = time.perf_counter()
        await asyncio.sleep(0.1)                               # cede el loop
        ahora = time.perf_counter()
        marcas.append(ahora)                                   # NUEVO: instante de cada latido
        registra("latido", antes, antes + 0.1, "espera")       # NUEVO: el sleep
        if ahora - antes - 0.1 > 0.01:
            registra("latido", antes + 0.1, ahora, "lista")   # NUEVO: venció y no tiene hilo
        registra("latido", ahora, ahora + 0.004, "exec")       # NUEVO: imprime

async def en_el_loop():
    await asyncio.sleep(0.15)
    ini = time.perf_counter()
    calcula()                                                  # sin await: el loop queda ocupado
    registra("cálculo", ini, time.perf_counter(), "exec")
    return time.perf_counter() - ini

async def en_un_hilo():
    await asyncio.sleep(0.15)
    ini = time.perf_counter()
    await asyncio.to_thread(calcula)                           # NUEVO: otro hilo del mismo proceso (mismo GIL)
    registra("cálculo", ini, time.perf_counter(), "exec")
    return time.perf_counter() - ini

async def en_un_proceso(pool):
    await asyncio.sleep(0.15)
    ini = time.perf_counter()
    await asyncio.get_running_loop().run_in_executor(pool, calcula)    # NUEVO: otro proceso (otro GIL)
    registra("cálculo", ini, time.perf_counter(), "exec")
    return time.perf_counter() - ini

async def corrida(nombre, trabajo, duracion=3.2):
    reinicia()
    marcas = []
    _, dur = await asyncio.gather(latido_g(duracion, marcas), trabajo)
    huecos = [b - a for a, b in zip(marcas, marcas[1:])]
    dibuja(f"{nombre} · intervalo máximo entre latidos {max(huecos):.2f} s")
    return dur, max(huecos)

with ProcessPoolExecutor(max_workers=1) as pool:
    pool.submit(calcula, 1).result()                           # NUEVO: arranca el proceso antes de medir
    filas = [("en el loop",) + await corrida("cálculo EN EL LOOP", en_el_loop()),
             ("en un hilo",) + await corrida("cálculo en un HILO", en_un_hilo()),
             ("en un proceso",) + await corrida("cálculo en un PROCESO", en_un_proceso(pool))]
print(f"{'dónde corre':<15}{'cálculo (s)':>12}{'mayor hueco entre latidos (s)':>32}")
for nombre, dur, hueco in filas:
    print(f"{nombre:<15}{dur:>12.2f}{hueco:>32.2f}")
CIFRAS.update(hueco_loop=filas[0][2], hueco_hilo=filas[1][2], hueco_proceso=filas[2][2], calc_s=filas[0][1])
assert filas[0][2] > 0.8 * filas[0][1], "con el cálculo en el loop, el hueco es casi todo el cálculo"
assert filas[1][2] < 0.5 and filas[2][2] < 0.3, "en un hilo o un proceso el loop sigue latiendo"

## 6.2 · El mismo latido, con el cálculo en un proceso

`run_in_executor(pool, calcula)` manda `calcula` (por nombre, con `pickle`) a un proceso hijo y devuelve un
futuro; el `await` suspende a `calculo_fuera` y **libera el hilo del loop**, así que `latido` sigue. Se pasa
`calcula`, no `calcula()`: con paréntesis se ejecutaría aquí, en el loop.

La celda reutiliza `latido` y `t0` de la 6.1.

**Qué esperar.** Los cinco latidos cada 0.1 s; el cálculo tarda un poco más que en el loop porque incluye
arrancar el proceso (el pool lo crea en el primer envío), mandar la tarea y traer el resultado.

In [ ]:
# celda 6.2
import asyncio
import time
from concurrent.futures import ProcessPoolExecutor
from trabajo_computo import calcula

async def calculo_fuera(pool):
    await asyncio.sleep(0.15)                      # el mismo arranque que en 6.1
    loop = asyncio.get_running_loop()              # el loop que corre esta celda
    inicio = time.perf_counter()
    await loop.run_in_executor(pool, calcula)      # calcula en otro proceso; el loop sigue
    print(f"      el cálculo duró {time.perf_counter() - inicio:.2f} s en otro proceso")

t0 = time.perf_counter()
with ProcessPoolExecutor(max_workers=1) as pool:   # un proceso hijo, con su propio GIL
    await asyncio.gather(latido(5), calculo_fuera(pool))

## 6.3 · Amdahl: lo que dice la fórmula

Con una fracción serial $f$ (lo que no se puede repartir) y $N$ trabajadores, el speedup máximo es
$S(N)=\dfrac{1}{f+(1-f)/N}$ y tiene techo $1/f$. La celda 6.3 es la fórmula tal cual: no mide nada.

In [ ]:
# celda 6.3
def amdahl(f, N):
    """Speedup con fracción serial f y N trabajadores (reparto perfecto)."""
    return 1 / (f + (1 - f) / N)

for f in (0.10, 0.25, 0.50):
    fila = [f"{amdahl(f, N):.2f}" for N in (2, 4, 8)]     # S(2), S(4), S(8)
    print(f"f={f:.2f} {fila} techo: {1 / f:.1f}")
print("f=0.25, de 4 a 8 trabajadores:", round(amdahl(0.25, 8) / amdahl(0.25, 4), 2), "veces más")

### 6.3 ampliada: medir 1, 2 y 4 procesos, y ajustar $f$

**Qué se mide.** Cuatro tareas independientes, cada una `calcula(5_000_000)` (≈ 0.3 s), repartidas con
`ProcessPoolExecutor(max_workers=N)` para $N=1,2,4$. El tiempo **incluye** crear el pool y arrancar los
procesos hijo, y eso es parte de la fracción serial: nadie reparte el arranque.

**Qué se calcula.**

- Speedup $S(N)=T_1/T_N$ y eficiencia $E(N)=S(N)/N$.
- $f$ **ajustada**: de $S=\dfrac{1}{f+(1-f)/N}$ se despeja $f=\dfrac{1/S-1/N}{1-1/N}$ para cada $N>1$ y se promedia.
- La curva $S(N)$ con esa $f$, contra los puntos medidos.

**Por qué no pasar de 4.** Hay sólo 4 tareas: con más de 4 trabajadores, los demás no tienen qué hacer. Es un
techo distinto del de Amdahl (reparto, no parte serial) y la fórmula no lo sabe. Tampoco conviene tener más
procesos que CPU lógicas $P$ (esta máquina tiene 8).

**Qué esperar.** $S(2)$ cerca de 1.6–1.9, $S(4)$ cerca de 2.3–3.3 (el arranque de los procesos es una fracción
fija; mientras menos dure el cálculo, más pesa). La $f$ ajustada es de unas décimas.

In [ ]:
# Cuatro tareas con 1, 2 y 4 procesos: tiempo, speedup, eficiencia, f ajustada y la curva.
N_TAREAS, N_CALC = 4, 5_000_000

def corre_procesos(N):
    t0 = time.perf_counter()
    with ProcessPoolExecutor(max_workers=N) as pool:           # el pool nace aquí: su arranque cuenta
        resultados = list(pool.map(calcula, [N_CALC] * N_TAREAS))   # NUEVO: 4 tareas repartidas entre N procesos
    return time.perf_counter() - t0, resultados

T = {}
for N in (1, 2, 4):
    T[N], res = corre_procesos(N)
    assert len(set(res)) == 1                                  # las 4 tareas dan el mismo resultado

S = {N: T[1] / T[N] for N in T}                                # NUEVO: speedup medido
E = {N: S[N] / N for N in T}                                   # NUEVO: eficiencia
fs = [(1 / S[N] - 1 / N) / (1 - 1 / N) for N in (2, 4)]        # NUEVO: f despejada de cada medición
f_ajustada = sum(fs) / len(fs)
print(f"{'N':>3}{'T (s)':>8}{'S medido':>10}{'S(N) con f':>12}{'E':>7}")
for N in T:
    print(f"{N:>3}{T[N]:>8.2f}{S[N]:>10.2f}{amdahl(f_ajustada, N):>12.2f}{E[N]:>7.2f}")
print(f"f ajustada = {f_ajustada:.2f}  (de N=2: {fs[0]:.2f}, de N=4: {fs[1]:.2f}) · techo 1/f = {1 / f_ajustada:.1f}")

xs = range(1, 9)
fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.plot(xs, list(xs), "--", color="#999999", label="ideal S = N")
ax.plot(xs, [amdahl(f_ajustada, n) for n in xs], color="#1f4e8c", label=f"Amdahl con f = {f_ajustada:.2f}")
ax.axhline(1 / f_ajustada, color="#8a5a00", linestyle=":", label=f"techo 1/f = {1 / f_ajustada:.1f}")
ax.plot(list(S), list(S.values()), "o", color="#c0392b", label="medido (4 tareas)")
ax.set_xlabel("procesos N"); ax.set_ylabel("speedup"); ax.set_title("La curva de Amdahl con la f de esta máquina")
ax.legend(); fig.tight_layout(); plt.show()
CIFRAS.update(T1=T[1], T2=T[2], T4=T[4], S2=S[2], S4=S[4], f=f_ajustada)
assert S[2] > 1.3 and S[4] > S[2], "más procesos deben acelerar"
assert S[4] < 4.0, "el speedup de 4 procesos no puede ser el ideal: el arranque es serial"
assert 0 < f_ajustada < 0.7

### Dónde corrió cada tarea: el Gantt de 4 tareas con 2 procesos

`calcula_medido` devuelve cuándo empezó y terminó **dentro del proceso hijo**, con `time.time()` (el reloj de
pared del sistema; los dos procesos lo comparten, el `perf_counter` de uno no es el del otro). El Gantt tiene
una fila por tarea: `lista` (rayado) de que se envió al pool hasta que algún trabajador la tomó, y `exec`
mientras calcula. Con 2 procesos y 4 tareas hay dos olas; la primera empieza **después** del envío porque
primero hay que arrancar los procesos.

**Qué esperar.** Las tareas 0 y 1 con `lista` corto (el arranque del proceso) y `exec` de ≈ 0.3 s; las tareas 2 y 3
con un `lista` largo (esperan a que termine la ola anterior).

In [ ]:
reinicia()
with ProcessPoolExecutor(max_workers=2) as pool:
    envio = time.time()
    futuros = [pool.submit(calcula_medido, N_CALC) for _ in range(N_TAREAS)]   # las 4 salen a la vez
    medidas = [f.result() for f in futuros]
for i, (ini, fin, _) in enumerate(medidas):
    registra(f"tarea {i}", envio, ini, "lista")                # esperó trabajador (o arranque del proceso)
    registra(f"tarea {i}", ini, fin, "exec")                   # calculó en un proceso hijo
dibuja("4 tareas, 2 procesos · rayado = en la cola del pool, relleno = calculando")
esperas = [ini - envio for ini, _, _ in medidas]
print("espera en la cola (s):", [round(e, 2) for e in esperas])
assert max(esperas[:2]) < min(esperas[2:]), "las dos primeras tareas arrancan antes que las dos últimas"

## Una `lambda` a un proceso

`pickle` serializa una función **por nombre**: módulo + nombre. `trabajo_computo.calcula` tiene uno que el hijo
puede importar. Una `lambda` se llama `<lambda>` y no es atributo de ningún módulo, así que no hay nombre que
mandar. El error se ve al serializar; aquí se ve **sin pool**, con `pickle.dumps`, para no depender del
momento en que el pool lo reporta (el tema 8 lo hace con el pool).

In [ ]:
bytes_calcula = pickle.dumps(calcula)
print("pickle.dumps(calcula):", len(bytes_calcula), "bytes · contiene el módulo:", b"trabajo_computo" in bytes_calcula)
try:
    pickle.dumps(lambda x: x * 2)
except Exception as e:
    print(f"pickle.dumps(lambda) -> {type(e).__name__}: {str(e)[:80]}")
    error = e
assert b"trabajo_computo" in bytes_calcula and b"calcula" in bytes_calcula
assert isinstance(error, (pickle.PicklingError, AttributeError))

## Cierre del tema: la tabla

| pregunta | respuesta medida |
|---|---|
| ¿un cálculo en el loop detiene las otras tareas? | sí: el mayor hueco entre latidos es casi toda la duración del cálculo |
| ¿un hilo lo arregla? | el loop sigue, pero el cálculo en Python puro no se acelera (mismo GIL) |
| ¿un proceso lo arregla? | sí: el loop sigue y el cálculo usa otra CPU |
| ¿cuánto acelera repartir entre 2 y 4 procesos? | menos que el ideal; la diferencia es la fracción serial $f$ |
| ¿cómo viaja la función a otro proceso? | por nombre: una `lambda` no tiene uno |

In [ ]:
print(f"{'cifra':<16}{'valor':>8}")
for nombre, valor in CIFRAS.items():
    print(f"{nombre:<16}{valor:>8.2f}")
assert CIFRAS["hueco_loop"] > 10 * CIFRAS["hueco_proceso"]
assert CIFRAS["T4"] < CIFRAS["T2"] < CIFRAS["T1"]
print("tema 6: todas las comprobaciones pasan")

# Tema 7 · Vectorizar: empujar el cálculo al motor

**Pregunta.** Antes de repartir un cálculo entre procesos, ¿se puede hacer que cada operación cueste menos?

Un bucle de Python hace, por cada elemento, trabajo de intérprete: busca el objeto, comprueba su tipo, llama a
`__mul__`, crea otro objeto `float`. **Vectorizar** es entregar el arreglo completo a una función compilada que
hace esa misma cuenta sobre valores crudos y contiguos en memoria. El cálculo sigue en **una** CPU, con **un**
proceso: no hay paralelismo ni concurrencia, y aun así es 10 a 100 veces más rápido. Por eso es lo primero que
se prueba, antes de los procesos del tema 6.

Lo que este tema agrega a la página 7:

| bloque | qué enseña |
|---|---|
| 7.1 y ampliación | qué ocupa un arreglo y por qué un tipo único lo hace barato |
| 7.2 y ampliación | cuánto gana NumPy según el tamaño, y qué cuesta un temporal |
| 7.3 y ampliación | `int64` se desborda **sin avisar**; qué tipo usar |
| vistas | qué copia NumPy y qué comparte: cortes, vistas no contiguas, transpuestas |

Todo el tema es local: no usa el servidor. Cargas: arreglos de hasta 5 millones de elementos (40 MB).

In [ ]:
# Preparación del tema.
import sys, time, tracemalloc
import numpy as np

CIFRAS = {}
print("numpy", np.__version__)

## 7.1 · Un arreglo de `float64`

Un `ndarray` guarda **un tipo único** (`dtype`) y los valores crudos uno detrás de otro. Por eso su tamaño es
`nbytes = n · itemsize`: 5 000 000 × 8 = 40 000 000 bytes. Una lista de Python con 5 millones de `float` guarda
**punteros** (8 bytes cada uno) a objetos `float` sueltos (24 bytes cada uno), repartidos por toda la memoria.

**Qué esperar.** `float64`, 8 bytes por elemento, 40 000 000 bytes, contiguo (`True`), y el tiempo de crear el
arreglo (unas pocas decenas de milisegundos).

In [ ]:
# celda 7.1
import time
import numpy as np

n = 5_000_000
t0 = time.perf_counter()
a = np.arange(n, dtype=np.float64)   # crea los valores ya como decimales de 64 bits
crear_s = time.perf_counter() - t0   # la creación se mide aparte

print("dtype:", a.dtype, "| bytes por elemento:", a.itemsize)
print("bytes totales:", a.nbytes)
print("contiguo:", a.flags["C_CONTIGUOUS"])
print(f"crear el arreglo: {crear_s:.4f} s")

### 7.1 ampliada: el mismo arreglo en cuatro tipos, y contra una lista

El tipo decide cuánto ocupa y cuánto cabe sin perder precisión. Se compara el mismo `arange` en `int32`,
`int64`, `float32` y `float64`, y se mide qué ocupa la lista equivalente: `sys.getsizeof(lista)` cuenta los
punteros, y cada `float` cuenta 24 bytes aparte (`sys.getsizeof(1.0)`).

**Qué esperar.** `int32`/`float32` ocupan la mitad que los de 64 bits. La lista ocupa **unas cuatro veces**
lo que ocupa el arreglo de `float64`: 8 bytes de puntero + 24 de objeto por elemento contra 8.

In [ ]:
# Mismo arreglo en cuatro tipos, y la lista equivalente.
filas = []
for tipo in (np.int32, np.int64, np.float32, np.float64):
    arr = np.arange(n, dtype=tipo)                                 # NUEVO: variar el dtype
    filas.append((np.dtype(tipo).name, arr.itemsize, arr.nbytes))
print(f"{'dtype':<10}{'bytes/elem':>12}{'MB':>10}")
for nombre, ib, nb in filas:
    print(f"{nombre:<10}{ib:>12}{nb / 1e6:>10.1f}")

lista = a.tolist()                                                 # NUEVO: la lista equivalente
bytes_lista = sys.getsizeof(lista) + len(lista) * sys.getsizeof(1.0)
print(f"{'lista':<10}{'8 + 24':>12}{bytes_lista / 1e6:>10.1f}   (punteros + objetos float)")
print(f"la lista ocupa {bytes_lista / a.nbytes:.1f} veces lo que el arreglo float64")
CIFRAS.update(mb_float64=a.nbytes / 1e6, mb_lista=bytes_lista / 1e6)
assert a.nbytes == n * 8 == 40_000_000 and a.flags["C_CONTIGUOUS"]
assert bytes_lista > 3 * a.nbytes

## 7.2 · El bucle contra la operación vectorizada

Las dos funciones calculan $\sum x_i^2$. `bucle` recorre la lista elemento por elemento; `vec` eleva al
cuadrado todo el arreglo y lo suma, en dos llamadas a código compilado. Las dos suman en **distinto orden**,
y los decimales no son asociativos, por eso se comparan con `np.isclose` y tolerancia relativa, no con `==`.

Detalles de medición: `tolist()` queda **fuera** del cronómetro (el bucle necesita su entrada) y `vec(a)` se corre
una vez antes de medir, porque la primera llamada paga costos de arranque.

**Qué esperar.** El bucle tarda decenas o cientos de milisegundos; NumPy, unos cuantos milisegundos. La
ganancia en esta máquina fue de ≈ 16× en la corrida de referencia; la tuya cambia.

In [ ]:
# celda 7.2
import time
import numpy as np

a = np.arange(5_000_000, dtype=np.float64)
lista = a.tolist()  # la conversión queda fuera de la medición

def bucle(xs):
    total = 0.0
    for x in xs:
        total += x * x
    return total

def vec(arr):
    return float(np.sum(arr * arr))

vec(a)  # calentamiento: la primera corrida se descarta
t0 = time.perf_counter(); r1 = bucle(lista); bucle_s = time.perf_counter() - t0
t0 = time.perf_counter(); r2 = vec(a); numpy_s = time.perf_counter() - t0
print(f"bucle {bucle_s:.3f} s | NumPy {numpy_s:.4f} s | {bucle_s/numpy_s:.0f}x")
print("coinciden:", np.isclose(r1, r2, rtol=1e-9))

### 7.2 ampliada: la ganancia según el tamaño, y lo que cuesta un temporal

**1. Barrido de tamaños.** Se repite la medición con $n=10^5, 10^6, 5\times10^6$, tomando el **mejor de 3** para
NumPy (reduce ruido; el bucle es lo bastante lento como para medirse una vez). La ganancia **baja** al crecer $n$: con
pocos elementos todo cabe en la caché de la CPU; con millones, el arreglo la rebasa y la memoria pasa a ser el límite (explicación probable; esta celda no la mide).

**2. El temporal.** `arr * arr` crea un arreglo nuevo de 40 MB antes de que `np.sum` lo recorra: la expresión
necesita memoria extra del tamaño de los datos. `np.dot(arr, arr)` calcula la misma suma de cuadrados sin
temporal. Se mide el **pico de memoria** de cada forma con `tracemalloc` (NumPy informa sus asignaciones).

**Qué esperar.** La ganancia va de ≈ 30× con $10^5$ elementos a ≈ 15× con $5\times10^6$ (en cualquier caso, muy por encima de 1). Pico de `np.sum(arr * arr)` ≈ 40 MB; pico de
`np.dot` ≈ 0 MB.

In [ ]:
# Barrido de tamaños (ganancia) y pico de memoria con y sin temporal.
def mejor_de(f, veces=3):
    mejor = float("inf")
    for _ in range(veces):
        t0 = time.perf_counter(); f(); mejor = min(mejor, time.perf_counter() - t0)
    return mejor

print(f"{'n':>10}{'bucle (s)':>11}{'NumPy (s)':>11}{'ganancia':>10}")
ganancias = []
for m in (100_000, 1_000_000, 5_000_000):                           # NUEVO: tres tamaños
    arr_m = np.arange(m, dtype=np.float64)
    lista_m = arr_m.tolist()
    t_bucle = mejor_de(lambda: bucle(lista_m), veces=1)
    t_vec = mejor_de(lambda: vec(arr_m))
    ganancias.append(t_bucle / t_vec)
    print(f"{m:>10,}{t_bucle:>11.4f}{t_vec:>11.5f}{t_bucle / t_vec:>9.0f}x")

def pico(f):
    tracemalloc.start()
    f()
    _, pico_bytes = tracemalloc.get_traced_memory()                 # NUEVO: el máximo de memoria asignada
    tracemalloc.stop()
    return pico_bytes

pico_temporal = pico(lambda: np.sum(a * a))                         # a*a crea un arreglo de 40 MB
pico_dot = pico(lambda: np.dot(a, a))                               # sin temporal
print(f"pico de memoria: np.sum(a*a) = {pico_temporal / 1e6:.1f} MB · np.dot(a, a) = {pico_dot / 1e6:.3f} MB")
print("mismo resultado:", np.isclose(np.sum(a * a), np.dot(a, a), rtol=1e-9))
CIFRAS.update(ganancia_5M=ganancias[-1], pico_temporal_mb=pico_temporal / 1e6, pico_dot_mb=pico_dot / 1e6)
assert np.isclose(r1, r2, rtol=1e-9)
assert all(g > 3 for g in ganancias), "NumPy debe ganar en los tres tamaños"
assert pico_temporal > 30e6 and pico_dot < 1e6

## 7.3 · `int64` se desborda sin avisar

Un `int64` guarda enteros de $-2^{63}$ a $2^{63}-1$ ($\approx9.2\times10^{18}$). La suma exacta de los
cuadrados de 0 a $n-1$ es $\dfrac{(n-1)\,n\,(2n-1)}{6}$; con $n=5\,000\,000$ vale $\approx4.17\times10^{19}$,
más de cuatro veces el máximo. NumPy **no lanza error**: el resultado «da la vuelta» y es otro número. `float64`
conserva la magnitud pero pierde los últimos dígitos (tiene 53 bits de mantisa). El `int` de Python no tiene
tope y sirve de referencia exacta.

**Qué esperar.** El exacto de Python, un `int64` con otro valor (posiblemente negativo) y un `float64` con
el orden de magnitud correcto en notación científica.

In [ ]:
# celda 7.3
import numpy as np

n = 5_000_000
enteros = np.arange(n, dtype=np.int64)
con_int64 = int(np.sum(enteros * enteros))           # se desborda sin avisar
exacto = (n - 1) * n * (2 * n - 1) // 6              # fórmula cerrada, enteros de Python
con_float64 = float(np.sum(np.arange(n, dtype=np.float64) ** 2))

print("máximo de int64:", np.iinfo(np.int64).max)
print("exacto (Python):", exacto)
print("int64 (NumPy):  ", con_int64)
print("float64 (NumPy):", f"{con_float64:.6e}")

### 7.3 ampliada: dónde empieza el desborde y qué hace el `float64`

Se barren valores de $n$ y se compara cada suma con la exacta. El primer $n$ que desborda es el que cumple
$(n-1)\,n\,(2n-1)/6 > 2^{63}-1$, alrededor de $3.04\times10^6$. Además, dos formas de ver el **mismo** problema
con un solo elemento: el escalar avisa (`RuntimeWarning`), el arreglo no.

**Qué esperar.** Para $n\le3\times10^6$ el `int64` coincide con el exacto; para $n\ge4\times10^6$ no. El
`float64` queda a menos de $10^{-12}$ del exacto (error relativo); en este rango la columna sale en 0 o casi.

In [ ]:
# Barrido de n: ¿desde dónde se desborda int64? ¿cuánto se equivoca float64?
tope = np.iinfo(np.int64).max
print(f"{'n':>12}{'exacto > tope':>15}{'int64 coincide':>16}{'error relativo float64':>26}")
desborda_en = None
for m in (1_000_000, 3_000_000, 4_000_000, 5_000_000):
    exacto_m = (m - 1) * m * (2 * m - 1) // 6
    s_int = int(np.sum(np.arange(m, dtype=np.int64) ** 2))
    s_flt = float(np.sum(np.arange(m, dtype=np.float64) ** 2))
    rel = abs(s_flt - exacto_m) / exacto_m
    print(f"{m:>12,}{str(exacto_m > tope):>15}{str(s_int == exacto_m):>16}{rel:>26.1e}")
    if desborda_en is None and s_int != exacto_m:
        desborda_en = m
    assert (s_int == exacto_m) == (exacto_m <= tope), "int64 coincide exactamente cuando el exacto cabe"
    assert rel < 1e-12

import warnings
with warnings.catch_warnings(record=True) as avisos:
    warnings.simplefilter("always")
    escalar = np.int64(tope) + np.int64(1)                          # NUEVO: el escalar avisa
    arreglo = np.array([tope], dtype=np.int64) + 1                  # NUEVO: el arreglo no avisa
print(f"escalar: {escalar} · avisos: {len(avisos)} ({avisos[0].category.__name__ if avisos else '-'})")
print(f"arreglo: {arreglo[0]} · da la vuelta al mínimo de int64: {arreglo[0] == np.iinfo(np.int64).min}")
CIFRAS["n_primer_desborde"] = desborda_en
assert desborda_en == 4_000_000
assert arreglo[0] == np.iinfo(np.int64).min, "el arreglo se desborda en silencio"
assert con_int64 != exacto and con_float64 > 4e19

## Vistas, copias y memoria no contigua

NumPy tiene dos formas de entregar «una parte» de un arreglo:

| operación | qué devuelve | cuesta |
|---|---|---|
| `a[100:200]` (corte simple) | **vista**: mismos datos, otros límites | casi nada |
| `a[::8]` (corte con paso) | **vista no contigua**: salta de 8 en 8 elementos (`strides`) | casi nada, pero recorrerla es más lento |
| `a[[1, 5, 9]]` (índices) | **copia** | O(tamaño) |
| `a.T` de una matriz | **vista** con `strides` invertidos (ya no es contigua por filas) | casi nada |
| `a.ravel()` | vista si es contigua; **copia** si no | depende |

Una vista comparte memoria con el original: **escribir en ella escribe en él**. Y una vista no contigua rompe la
propiedad que hacía rápido a NumPy (valores consecutivos en memoria, que la CPU trae en bloque): se pagan más
fallos de caché.

**Qué esperar.** `shares_memory` verdadero para las vistas y falso para la copia; `C_CONTIGUOUS` falso para
`a[::8]` y `.T`; modificar la vista cambia el original; sumar la vista `[::8]` (un octavo de los
datos) cuesta bastante más que un octavo del tiempo de sumar todo, porque cada línea de caché de 64 bytes trae 8 `float64` y la
vista usa uno solo por línea; la copia contigua del mismo octavo es la más rápida.

In [ ]:
# Vistas, copias y contigüidad.
b = np.arange(1_000_000, dtype=np.float64)
corte = b[100:200]                                  # vista contigua
paso = b[::8]                                       # vista NO contigua: una de cada 8
copia = b[[1, 5, 9]]                                # copia
print("corte  → comparte memoria:", np.shares_memory(b, corte), "· contiguo:", corte.flags["C_CONTIGUOUS"], "· base es b:", corte.base is b)
print("paso   → comparte memoria:", np.shares_memory(b, paso), "· contiguo:", paso.flags["C_CONTIGUOUS"], "· strides:", paso.strides)
print("copia  → comparte memoria:", np.shares_memory(b, copia), "· contiguo:", copia.flags["C_CONTIGUOUS"])

corte[0] = -1.0                                     # escribir en la vista
print("b[100] después de escribir en la vista:", b[100])

m = np.arange(3000 * 3000, dtype=np.float64).reshape(3000, 3000)
t = m.T                                             # transpuesta: vista con los strides invertidos
print("m.T → comparte memoria:", np.shares_memory(m, t), "· contiguo por filas:", t.flags["C_CONTIGUOUS"], "· strides:", t.strides)
print("m.ravel() comparte memoria:", np.shares_memory(m, m.ravel()), "· t.ravel() comparte memoria:", np.shares_memory(m, t.ravel()))

# Costo de recorrer: todo b, un octavo de b con paso 8 (vista), y ese octavo ya copiado contiguo.
grande = np.arange(40_000_000, dtype=np.float64)    # 320 MB: bastante más que la caché
vista8 = grande[::8]
contiguo8 = vista8.copy()
t_todo = mejor_de(lambda: grande.sum())
t_vista = mejor_de(lambda: vista8.sum())
t_cont = mejor_de(lambda: contiguo8.sum())
print(f"sumar los 40 M:                    {t_todo * 1000:7.1f} ms")
print(f"sumar 1/8 por la vista no contigua: {t_vista * 1000:7.1f} ms")
print(f"sumar ese 1/8 ya copiado contiguo:  {t_cont * 1000:7.1f} ms")
del grande, vista8, contiguo8
CIFRAS.update(t_todo_ms=t_todo * 1000, t_vista8_ms=t_vista * 1000, t_cont8_ms=t_cont * 1000)
assert np.shares_memory(b, corte) and np.shares_memory(b, paso) and not np.shares_memory(b, copia)
assert not paso.flags["C_CONTIGUOUS"] and not t.flags["C_CONTIGUOUS"] and b[100] == -1.0
assert np.shares_memory(m, m.ravel()) and not np.shares_memory(m, t.ravel())
assert t_cont < t_vista, "la misma suma sobre memoria contigua es más rápida que sobre la vista"

## Cierre del tema: la tabla

| pregunta | respuesta medida |
|---|---|
| ¿cuánto ocupa un `float64` de 5 M? | 40 MB; la lista equivalente, cuatro veces más |
| ¿cuánto gana NumPy contra el bucle? | de 10× a 30× en esta máquina; baja al crecer $n$ porque el arreglo rebasa la caché |
| ¿qué cuesta una expresión como `a * a`? | un temporal del tamaño de los datos; `np.dot` no lo necesita |
| ¿cuándo se desborda `int64` en la suma de cuadrados? | desde $n=4\times10^6$; sin error ni aviso en arreglos |
| ¿una vista copia? | no: comparte memoria y escribir en ella cambia el original |
| ¿una vista con paso es tan rápida? | no: pierde contigüidad |

In [ ]:
print(f"{'cifra':<20}{'valor':>12}")
for nombre, valor in CIFRAS.items():
    print(f"{nombre:<20}{valor:>12.2f}")
assert CIFRAS["ganancia_5M"] > 5 and CIFRAS["mb_lista"] > 3 * CIFRAS["mb_float64"]
print("tema 7: todas las comprobaciones pasan")

# Tema 8 · Elegir la estrategia: una tabla de decisión que se ejecuta

**Pregunta.** Dado un trabajo —cuántas tareas, cuánto de cada una es CPU y cuánto es espera, qué tipo de
operación—, ¿qué estrategia se elige, y qué tiempo se predice para ella?

La página 8 resume la unidad en una tabla. Aquí la tabla es **código**: una función `decide` recibe los datos del
trabajo y devuelve la estrategia, y otra, `predice`, devuelve el tiempo que la teoría espera. Se alimentan con
**tus mediciones** (no con las de la página) y se comparan con una corrida real. Tres partes:

| parte | qué hace |
|---|---|
| 1 · Little | $L=\lambda\cdot W$ aplicada a tus 500 llamadas: cuántas hay en vuelo, cuántos hilos ocupa el servidor |
| 2 · la decisión | medir $\rho=e/d$ de dos trabajos, escribir `decide` y probarla en una tabla de casos |
| 3 · la `lambda` | por qué la herramienta que parecía bastar no puede viajar a un proceso, y tres formas de arreglarlo |

**Notación (la fija la página `A_notacion`).** $e$ = tiempo de CPU que la tarea ejecuta; $d$ = duración total de su
vida (de que empieza a que termina); $\rho=e/d$ ∈ [0, 1]. $\rho\approx0$: casi todo es espera. $\rho\approx1$:
casi todo es cálculo. $W$ = tiempo que pasa una petición en el sistema; $\lambda$ = peticiones terminadas por
segundo; $L$ = cuántas hay dentro, en promedio.

Cargas: 500 llamadas y 4 procesos como máximo. El servidor de la clase debe estar prendido.

In [ ]:
# Preparación del tema.
import asyncio, math, os, pickle, time
from concurrent.futures import ProcessPoolExecutor
from functools import partial
import httpx, requests
from cable import URL
from trabajo_computo import calcula

CIFRAS = {}
P = os.cpu_count()                  # CPU lógicas de esta máquina
print("P =", P, "CPU lógicas")

## 8.1 · Little con las cifras de la página

La Ley de Little dice que, en régimen estable, $L=\lambda\cdot W$. Con $k=32$ llamadas en vuelo
($L=k$) y $W=0.2$ s, el sistema no puede terminar más de $\lambda=k/W=160$ por segundo. De ahí salen la **cota**
$\lceil n/k\rceil\cdot W$ y, despejando, la $W$ *efectiva* que midió tu corrida:
$W_{\text{ef}}=L/\lambda=k/\lambda_{\text{medido}}$.

La celda 8.1 usa los **3.32 s** de la corrida de referencia. En la ampliación se reemplazan por los tuyos.

**Qué esperar.** $\lambda$ ideal 160/s, cota 3.20 s, $\lambda$ medido 151/s, $W\le0.212$ s (una cota: $L\le32$) y unos 30
hilos del servidor; si cada llamada tardara 2 s, 301.

In [ ]:
# celda 8.1
import math

k, W = 32, 0.2                  # tope en vuelo y segundos que tarda cada llamada
n, medido = 500, 3.32           # llamadas y segundos medidos (pon los tuyos)

lam_ideal = k / W               # Little con L = k: λ ≤ k / W
cota = math.ceil(n / k) * W     # ⌈n/k⌉ tandas de W segundos
lam_medido = n / medido         # peticiones terminadas por segundo
W_max = k / lam_medido          # L ≤ k, así que W = L / λ ≤ k / λ: una cota, no una medida

print(f"λ ideal  {lam_ideal:.0f}/s   cota  {cota:.2f} s")
print(f"λ medido {lam_medido:.0f}/s   W ≤ {W_max:.3f} s")
print(f"hilos del servidor: {lam_medido * W:.0f}; con W = 2 s: {lam_medido * 2:.0f}")

### 8.1 ampliada: tus 500 llamadas, y los hilos del servidor **contados**

En lugar de suponer 3.32 s se corre el lote de 500 llamadas con $k=32$ (`Semaphore(32)` +
`Limits(32)`, como en el tema 4) y se usa `GET /quien`, que responde **el puerto de la conexión y el hilo del
servidor** que la atiende. Contar los hilos distintos es contar cuántos hilos del servidor usó el cliente: la
predicción de Little ($\lambda\cdot W\approx30$) es un promedio; el máximo observable es $k=32$ (una conexión por
llamada en vuelo, un hilo por conexión).

**Qué esperar.** Un tiempo cercano a la cota, una $W_{\text{ef}}$ entre 0.20 y 0.23 s, y a lo más 32 hilos del
servidor distintos.

In [ ]:
# NUEVO: tu propia medición de 500 llamadas, con /quien para contar hilos del servidor.
async def lote_little(n, k):
    sem = asyncio.Semaphore(k)
    async def una(cliente):
        async with sem:
            return (await cliente.get(f"{URL}/quien?lento=0.2")).json()
    async with httpx.AsyncClient(limits=httpx.Limits(max_connections=k), timeout=30) as cliente:
        t0 = time.perf_counter()
        vistos = await asyncio.gather(*(una(cliente) for _ in range(n)))
        return time.perf_counter() - t0, vistos

k, W, n = 32, 0.2, 500
medido, vistos = await lote_little(n, k)                    # NUEVO: tu medido, no el de la página
hilos_distintos = len({v["hilo"] for v in vistos})
lam_ideal, cota = k / W, math.ceil(n / k) * W
lam_medido = n / medido
W_efectiva = k / lam_medido
print(f"medido {medido:.2f} s · cota {cota:.2f} s")
print(f"λ ideal {lam_ideal:.0f}/s · λ medido {lam_medido:.0f}/s · W efectiva {W_efectiva:.3f} s")
print(f"hilos del servidor por Little: {lam_medido * W:.0f} · contados con /quien: {hilos_distintos} (máximo posible {k})")
print(f"con W = 2 s la predicción sería {lam_medido * 2:.0f} hilos: una dependencia 10 veces más lenta ocupa 10 veces más")
CIFRAS.update(medido_s=medido, lam_medido=lam_medido, W_ef=W_efectiva, hilos=hilos_distintos)
assert cota * 0.98 <= medido < cota * 1.5
assert W <= W_efectiva < W * 1.25, "la W efectiva no puede ser menor que la W del servidor"
assert 1 <= hilos_distintos <= k

## 8.2 · Una `lambda` a un proceso

`pool.submit(f, x)` **no** ejecuta `f` en tu línea: la mete en una cola, un hilo auxiliar del pool la serializa con
`pickle` y la manda a un proceso hijo. `pickle` serializa una función **por nombre** (módulo + nombre) y el hijo
la busca importando ese módulo. Una `lambda` se llama `<lambda>` y no es atributo de ningún módulo: no hay nombre
que mandar. El error aparece en `futuro.result()`, no en `submit`.

**Qué esperar.** Un `PicklingError` («Can't pickle … it's not found as `__main__.<lambda>`»), y la tabla de
la página: `submit` pone la llamada en la cola, el envío falla, `result` relanza.

In [ ]:
# celda 8.2
from concurrent.futures import ProcessPoolExecutor

with ProcessPoolExecutor(max_workers=2) as pool:
    futuro = pool.submit(lambda x: x * 2, 21)   # la función viaja serializada con pickle
    try:
        print(futuro.result())                  # relanza aquí el error del envío
    except Exception as e:
        print(f"{type(e).__name__} · {e}")

### 8.2 ampliada: la `lambda` falla, tres formas de arreglarla

Se prueba cada forma con el **mismo** pool y el mismo cálculo, y se verifica el resultado:

| forma | por qué viaja |
|---|---|
| `lambda x: x * 2` | no viaja: sin nombre importable |
| `calcula` (función de `trabajo_computo.py`) | viaja: módulo `trabajo_computo` + nombre `calcula` |
| `partial(calcula, 1000)` | viaja: `partial` se serializa como la función por nombre + los argumentos fijos |
| `partial(operator.mul, 2)` | viaja: `operator.mul` es una función de la librería estándar, con nombre |

La regla práctica: **lo que se manda a un proceso vive en un módulo con nombre**. Un `def` dentro de otra función
(una *clausura*) tampoco viaja, por la misma razón. Y los argumentos también se serializan: un archivo abierto o
una conexión de red no.

In [ ]:
import operator
resultados = {}
with ProcessPoolExecutor(max_workers=2) as pool:
    casos = {
        "lambda": (lambda x: x * 2, (21,)),
        "función de módulo": (calcula, (1000,)),
        "partial(calcula, 1000)": (partial(calcula, 1000), ()),            # NUEVO: argumentos fijos con partial
        "partial(operator.mul, 2)": (partial(operator.mul, 2), (21,)),     # NUEVO: función de la librería estándar
    }
    for nombre, (f, args) in casos.items():
        futuro = pool.submit(f, *args)
        try:
            resultados[nombre] = futuro.result()
        except Exception as e:
            resultados[nombre] = e
print(f"{'forma':<28}resultado")
for nombre, r in resultados.items():
    texto = f"{type(r).__name__}" if isinstance(r, Exception) else str(r)
    print(f"{nombre:<28}{texto}")
CIFRAS["lambda_error"] = type(resultados["lambda"]).__name__ == "PicklingError"
assert isinstance(resultados["lambda"], (pickle.PicklingError, AttributeError))
assert resultados["función de módulo"] == calcula(1000) == resultados["partial(calcula, 1000)"]
assert resultados["partial(operator.mul, 2)"] == 42

## 8.3 · ρ medida: ¿de qué es el trabajo?

El primer paso para decidir es **medir** $\rho=e/d$ del trabajo. Se aproxima $e$ con el tiempo de CPU del proceso
(`time.process_time()`) y $d$ con el de pared (`time.perf_counter()`), mientras el trabajo corre solo:

- **A · 20 llamadas** con `requests` a `lento=0.2` (4 s de pared).
- **B · un cálculo** `calcula(5_000_000)` (menos de 1 s).

La ρ de A debe quedar por debajo de 0.05 (el cliente casi no usa CPU: 99 % de su vida es espera) y la de B cerca
de 1. Con esas dos cifras se alimenta la tabla de decisión.

**Qué esperar.** $\rho_A\approx0.01$ y $\rho_B\approx1.0$.

In [ ]:
def mide_rho(trabajo):
    cpu0, t0 = time.process_time(), time.perf_counter()
    trabajo()
    cpu, pared = time.process_time() - cpu0, time.perf_counter() - t0
    return cpu / pared, cpu, pared

rho_A, cpu_A, d_A = mide_rho(lambda: [requests.get(f"{URL}/ventas?lento=0.2", timeout=30) for _ in range(20)])
rho_B, cpu_B, d_B = mide_rho(lambda: calcula(5_000_000))
print(f"{'trabajo':<22}{'e = CPU (s)':>12}{'d = pared (s)':>15}{'ρ = e/d':>9}")
print(f"{'A · 20 llamadas':<22}{cpu_A:>12.3f}{d_A:>15.2f}{rho_A:>9.3f}")
print(f"{'B · calcula(5 M)':<22}{cpu_B:>12.3f}{d_B:>15.2f}{rho_B:>9.3f}")
CIFRAS.update(rho_A=rho_A, rho_B=rho_B)
assert rho_A < 0.05, "20 llamadas: casi toda su vida es espera"
assert rho_B > 0.9, "un bucle de Python puro: casi toda su vida es CPU"

## 8.4 · La tabla de decisión como función

`decide` recibe lo que se midió y devuelve una estrategia. Las ramas son las filas de la tabla de la página 8:

| $\rho$ | tipo de trabajo | $n$ | estrategia |
|---|---|---|---|
| cualquiera | arreglo numérico con operación vectorizable | cualquiera | **vectorizar** (NumPy / Polars / SQL), antes que repartir |
| cualquiera | cualquiera | 1 | **secuencial**: nada que repartir |
| $<0.1$ (espera) | librería asíncrona disponible | acotada | **`asyncio` + `Semaphore(k)`**, con $k$ del destino |
| $<0.1$ | sólo librería bloqueante | acotada | **hilos** (`ThreadPoolExecutor(k)`) |
| $<0.1$ | cualquiera | sin cota | **cola acotada + $k$ consumidores** (no `gather`) |
| $>0.9$ (CPU) | Python puro | $\ge2$ | **procesos** $\min(n,P)$; medir $f$ antes |
| intermedia | cualquiera | $\ge2$ | **separar las fases**: espera con `asyncio`, cálculo en procesos |

**Una advertencia sobre los umbrales.** 0.1 y 0.9 son los de este curso, no leyes de la naturaleza: sirven para
elegir con una medición y se revisan con la siguiente. La función los deja como parámetros.

In [ ]:
def decide(rho, tipo, n, *, async_disponible=True, sin_cota=False, umbral_espera=0.1, umbral_cpu=0.9):
    """Devuelve la estrategia para un trabajo con ρ = e/d, tipo ('vectorizable' | 'python'), n tareas."""
    if tipo == "vectorizable":
        return "vectorizar (NumPy / Polars / SQL)"
    if n <= 1:
        return "secuencial"
    if rho < umbral_espera:                                  # casi todo es espera
        if sin_cota:
            return "cola acotada + k consumidores"
        return "asyncio + Semaphore(k)" if async_disponible else "hilos (ThreadPoolExecutor)"
    if rho > umbral_cpu:                                     # casi todo es CPU
        return "procesos (min(n, P))"
    return "separar fases: asyncio para esperar, procesos para calcular"

def predice(estrategia, n, e, w, k=32, P=P):
    """Segundos que la teoría espera: e = CPU por tarea, w = espera por tarea (las dos en s)."""
    if estrategia == "secuencial" or n <= 1:
        return n * (e + w)
    if estrategia.startswith(("asyncio", "hilos", "cola")):
        return math.ceil(n / k) * (e + w)                    # tandas de k
    if estrategia.startswith("procesos"):
        return math.ceil(n / min(n, P)) * e                  # tandas de min(n, P)
    return float("nan")

casos = [  # (descripción, ρ, tipo, n, extras)
    ("20 llamadas a una API",               rho_A, "python",       20,   {}),
    ("500 llamadas, librería async",        rho_A, "python",       500,  {}),
    ("500 llamadas, sólo requests",         rho_A, "python",       500,  {"async_disponible": False}),
    ("10 000 URLs, sin cota",               rho_A, "python",       10_000, {"sin_cota": True}),
    ("4 cálculos de Python puro",           rho_B, "python",       4,    {}),
    ("1 cálculo",                           rho_B, "python",       1,    {}),
    ("suma de cuadrados de 5 M",            rho_B, "vectorizable", 1,    {}),
    ("descarga + cálculo por tarea",        0.5,   "python",       50,   {}),
]
print(f"{'caso':<34}{'ρ':>6}{'n':>8}  estrategia")
decisiones = {}
for desc, rho, tipo, n_caso, extra in casos:
    decisiones[desc] = decide(rho, tipo, n_caso, **extra)
    print(f"{desc:<34}{rho:>6.2f}{n_caso:>8}  {decisiones[desc]}")

assert decisiones["20 llamadas a una API"].startswith("asyncio")
assert decisiones["500 llamadas, sólo requests"].startswith("hilos")
assert decisiones["10 000 URLs, sin cota"].startswith("cola")
assert decisiones["4 cálculos de Python puro"].startswith("procesos")
assert decisiones["1 cálculo"] == "secuencial"
assert decisiones["suma de cuadrados de 5 M"].startswith("vectorizar")
assert decisiones["descarga + cálculo por tarea"].startswith("separar")

## 8.5 · La predicción contra una corrida real

Una tabla de decisión que no se contrasta con la realidad es una opinión. Se toman dos casos, se piden la
estrategia a `decide` y el tiempo a `predice`, y se **corre esa estrategia** para comparar:

| caso | estrategia elegida | predicción | medición |
|---|---|---|---|
| 500 llamadas, $e\approx0$, $w=0.2$ s | `asyncio + Semaphore(32)` | $\lceil500/32\rceil\cdot0.2=3.2$ s | la `medido` de la ampliación 8.1 |
| 4 cálculos de 5 M | `procesos` | $\lceil4/\min(4,P)\rceil\cdot e$ | 4 procesos reales |

La predicción es una **cota inferior**: ignora el arranque de los procesos y la latencia de cada llamada fuera del
`sleep` del servidor. La medición debe quedar *por encima* y cerca.

**Qué esperar.** Un cociente medido/predicho cerca de 1.1 para las llamadas y entre 1.5 y 3 para los procesos: con un
cálculo de ≈ 0.3 s en total, arrancar los procesos hijo pesa tanto como el cálculo mismo.

In [ ]:
est_red = decide(rho_A, "python", 500)
pred_red = predice(est_red, 500, e=0.0, w=0.2)
est_cpu = decide(rho_B, "python", 4)
e_tarea = cpu_B                                                    # CPU de una tarea, medida en 8.3
pred_cpu = predice(est_cpu, 4, e=e_tarea, w=0.0)

t0 = time.perf_counter()
with ProcessPoolExecutor(max_workers=min(4, P)) as pool:           # ≤ 4 procesos
    sumas = list(pool.map(calcula, [5_000_000] * 4))
medido_cpu = time.perf_counter() - t0
assert len(set(sumas)) == 1

print(f"{'caso':<26}{'estrategia':<26}{'predicho':>9}{'medido':>9}{'medido/pred':>12}")
print(f"{'500 llamadas':<26}{est_red:<26}{pred_red:>8.2f}s{medido:>8.2f}s{medido / pred_red:>12.2f}")
print(f"{'4 cálculos de 5 M':<26}{est_cpu:<26}{pred_cpu:>8.2f}s{medido_cpu:>8.2f}s{medido_cpu / pred_cpu:>12.2f}")
CIFRAS.update(pred_red=pred_red, pred_cpu=pred_cpu, medido_cpu=medido_cpu)
assert pred_red * 0.98 <= medido < pred_red * 1.6
assert pred_cpu * 0.9 <= medido_cpu < pred_cpu * 5.0, "el arranque de procesos explica la diferencia"

## Cierre del tema: la tabla de lo medido

| pregunta | respuesta medida |
|---|---|
| ¿cuántas llamadas hay en vuelo y cuántos hilos ocupa el servidor? | $L=k=32$; hilos ≤ 32 (Little da ≈ 30 en promedio) |
| ¿qué $W$ efectiva tuvo cada llamada? | un poco más que 0.2 s: el costo de ir y volver |
| ¿qué le pasa a una `lambda` en un proceso? | `PicklingError`: sin nombre importable |
| ¿$\rho$ de 20 llamadas? ¿de un bucle? | ≈ 0 y ≈ 1 |
| ¿qué estrategia sale de la función? | la de la tabla, y su predicción queda bajo la medición |

In [ ]:
print(f"{'cifra':<14}{'valor':>10}")
for nombre, valor in CIFRAS.items():
    print(f"{nombre:<14}{valor:>10.3f}" if isinstance(valor, float) else f"{nombre:<14}{str(valor):>10}")
assert CIFRAS["rho_A"] < 0.05 < 0.9 < CIFRAS["rho_B"]
assert CIFRAS["lambda_error"]
print("tema 8: todas las comprobaciones pasan")